In [ ]:
import pandas as pd

rte_grps = pd.read_csv('/Users/lucabutera/sumo-abq/cccar_dag_allday_route_groups.csv')

In [ ]:
rte_grps

In [ ]:
cntrd_splts = pd.read_csv("/Users/lucabutera/sumo-abq/cccar_dag_allday_centroid_splits.csv")

In [ ]:
cntrd_splts

In [ ]:
import numpy as np
import networkx as nx
import geopandas as gpd
import osmnx as ox
import folium

ox.settings.use_cache = True
ox.settings.log_console = True

# ----------------------------
# CONFIG (edit these)
# ----------------------------
place = "Bernalillo County, NM"
network_type = "drive"

# NOTE: you had y,x order in the comment but used y0,x0 variables.
# OSMnx expects X=lon, Y=lat.
y0, x0 = 35.089004161208564, -106.59095403952803   # start (lat, lon)
y1, x1 = 35.194257447325974, -106.66607311977107   # end   (lat, lon)

slack = 1.50                 # budget = slack * ssp
eps_rel = 1e-9               # float hygiene
mid_frac = 0.50              # "middle" at 50% of ssp along d_o
mid_band_width_frac = 0.06   # +/- 6% of ssp (small, stable; change if you want)
keep_dag_by_do = True        # apply d_o DAG filter to the SPT-union core
add_lateral = True
lateral_jump_frac = 0.06   # allow edges that advance <= 6% of ssp in d_o
lateral_keep_per_u = 8      # cap extra outgoing edges per node (keeps graph sane)
# ----------------------------

# ----------------------------
# 1) Load graph and pick nodes
# ----------------------------
G = ox.graph_from_place(place, network_type=network_type, simplify=True)

# Ensure travel_time exists
sample_edge = next(iter(G.edges(data=True)))[-1]
if "travel_time" not in sample_edge:
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)

orig = ox.nearest_nodes(G, X=x0, Y=y0)
dest = ox.nearest_nodes(G, X=x1, Y=y1)
print("Origin node:", orig, "Dest node:", dest)

# ----------------------------
# 2) Distances + SSP
# ----------------------------
dist_o = nx.single_source_dijkstra_path_length(G, orig, weight="travel_time")
dist_to_d = nx.single_source_dijkstra_path_length(G.reverse(copy=False), dest, weight="travel_time")

ssp_path = nx.shortest_path(G, orig, dest, weight="travel_time")
ssp = 0.0
for u, v in zip(ssp_path[:-1], ssp_path[1:]):
    ed = G.get_edge_data(u, v)
    best = min(d.get("travel_time", np.inf) for d in ed.values())
    ssp += float(best)

if not np.isfinite(ssp) or ssp <= 0:
    raise RuntimeError("SSP not finite/positive; choose different OD points.")

budget = slack * ssp
eps = eps_rel * ssp
print(f"SSP: {ssp:.1f}s   budget(slack={slack:.3f}): {budget:.1f}s")

# ----------------------------
# 3) Candidate nodes in band: d_o(u)+d_d(u)<=budget
# ----------------------------
cand_nodes = [
    u for u in G.nodes
    if np.isfinite(dist_o.get(u, np.inf))
    and np.isfinite(dist_to_d.get(u, np.inf))
    and (dist_o[u] + dist_to_d[u] <= budget)
]
cand_set = set(cand_nodes)
print("Candidate nodes:", len(cand_nodes), "of", G.number_of_nodes())

F = {u: dist_o[u] - dist_to_d[u] for u in cand_nodes}

def topo_index_of_multidigraph(H: nx.MultiDiGraph) -> dict:
    # topological_sort works on DiGraph; multiedges don't matter for acyclicity
    D = nx.DiGraph()
    D.add_nodes_from(H.nodes)
    D.add_edges_from((u, v) for u, v in H.edges())
    order = list(nx.topological_sort(D))
    return {n: i for i, n in enumerate(order)}

def fill_middle_edges_by_topo(
    H: nx.MultiDiGraph,
    H_gate: nx.MultiDiGraph,
    *,
    per_u_cap: int = 8,          # keep graph sane
    max_forward_span: int = 400, # optional: don't connect super-far in topo
    prefer_fast: bool = True,    # choose best travel_time key per (u,v)
) -> nx.MultiDiGraph:
    """
    Adds extra edges that already exist in H_gate, but only between nodes already in H,
    and only if they go forward in a fixed topological order of H.

    Complexity: O(m_gate_core) where m_gate_core = edges in H_gate induced on H.nodes,
    plus small per-node selection overhead.
    """
    if H.number_of_nodes() == 0:
        return H

    idx = topo_index_of_multidigraph(H)
    core = set(H.nodes)

    # candidates[u] = list of (score, u, v, key, data) where score sorts by desirability
    candidates = {u: [] for u in core}

    # Single scan over feasible (budget-gated) edges, but only among core nodes
    for u, v, k, data in H_gate.edges(keys=True, data=True):
        if u not in core or v not in core:
            continue

        iu, iv = idx[u], idx[v]
        if iv <= iu:
            continue  # topo-forward only -> DAG preserved

        if max_forward_span is not None and (iv - iu) > max_forward_span:
            continue

        if H.has_edge(u, v, key=k):
            continue

        tt = float(data.get("travel_time", np.inf))
        # rank candidates: prefer small travel_time; secondary prefer shorter topo span
        score = (tt, iv - iu)

        candidates[u].append((score, u, v, k, data))

    # Now pick up to per_u_cap edges per u.
    # If prefer_fast, keep only best key per (u,v) among candidates.
    for u, items in candidates.items():
        if not items:
            continue

        # Sort once; typical per-u lists are small-ish
        items.sort(key=lambda t: t[0])

        used_pairs = set()
        added = 0
        for (score, uu, vv, kk, dd) in items:
            if added >= per_u_cap:
                break

            if prefer_fast:
                pair = (uu, vv)
                if pair in used_pairs:
                    continue
                used_pairs.add(pair)

                # pick best key among H_gate(u,v)
                ed = H_gate.get_edge_data(uu, vv)
                if not ed:
                    continue
                best_k, best_tt, best_data = None, np.inf, None
                for k2, d2 in ed.items():
                    tt2 = float(d2.get("travel_time", np.inf))
                    if tt2 < best_tt:
                        best_tt, best_k, best_data = tt2, k2, d2
                if best_data is None or not np.isfinite(best_tt):
                    continue
                if not H.has_edge(uu, vv, key=best_k):
                    H.add_edge(uu, vv, key=best_k, **best_data)
                    added += 1
            else:
                H.add_edge(uu, vv, key=kk, **dd)
                added += 1

    return H

ARTERIAL_CLASSES = {
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
}

def highway_class(data) -> str:
    hw = data.get("highway")
    if isinstance(hw, (list, tuple)):
        return str(hw[0])
    if hw is None:
        return ""
    return str(hw)

def best_edge_key_data(Hg: nx.MultiDiGraph, u, v):
    ed = Hg.get_edge_data(u, v)
    if not ed:
        return None, None
    best_k, best_tt, best_data = None, np.inf, None
    for k, d in ed.items():
        tt = float(d.get("travel_time", np.inf))
        if tt < best_tt:
            best_tt, best_k, best_data = tt, k, d
    if best_data is None or not np.isfinite(best_tt):
        return None, None
    return best_k, best_data

# ----------------------------
# 4) Build core subgraphs (original rule vs "two-tree + middle band" technique)
# ----------------------------
def reachability_prune(H: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """Keep only nodes on some o->d path in H."""
    if orig not in H or dest not in H:
        return nx.MultiDiGraph()
    fwd = set(nx.descendants(H, orig)) | {orig}
    bwd = set(nx.descendants(H.reverse(copy=False), dest)) | {dest}
    core = fwd & bwd
    if orig not in core or dest not in core:
        return nx.MultiDiGraph()
    return H.subgraph(core).copy()

def add_nodes_with_attrs(H: nx.MultiDiGraph, nodes):
    H.add_nodes_from((n, G.nodes[n]) for n in nodes)

def build_core_original() -> nx.MultiDiGraph:
    """Your current gated + F(v)>F(u) + reachability prune. (Now preserves node attrs.)"""
    H = nx.MultiDiGraph()
    H.graph.update(G.graph)  # preserve CRS/metadata

    # IMPORTANT: include x/y node attrs so graph_to_gdfs can build geometries
    add_nodes_with_attrs(H, cand_nodes)

    for u in cand_nodes:
        du = dist_o[u]
        for v, edict in G[u].items():
            if v not in cand_set:
                continue
            hv = dist_to_d.get(v, np.inf)
            if not np.isfinite(hv):
                continue
            for k, data in edict.items():
                w_uv = float(data.get("travel_time", np.inf))
                if not np.isfinite(w_uv):
                    continue
                if du + w_uv + hv > budget:
                    continue
                if F[v] <= F[u] + eps:
                    continue
                H.add_edge(u, v, key=k, **data)

    return reachability_prune(H)

def build_core_two_tree_middleband() -> nx.MultiDiGraph:
    """
    Meet-in-the-middle on the *budget-gated* subgraph, so parents stay inside the core.
    """

    # --- A) Build the slack-gated subgraph on cand_set (NO monotone F filter) ---
    H_gate = nx.MultiDiGraph()
    H_gate.graph.update(G.graph)
    add_nodes_with_attrs(H_gate, cand_nodes)

    print("H_gate out-degree orig:", H_gate.out_degree(orig))
    print("H_gate in-degree dest:", H_gate.in_degree(dest))
    print("H_gate out-degree dest:", H_gate.out_degree(dest))
    print("H_gate in-degree orig:", H_gate.in_degree(orig))

    # Also check a few outgoing edges from orig in the full G and see why they fail
    cnt = 0
    for v, edict in G[orig].items():
        if v not in cand_set:
            continue
        hv = dist_to_d.get(v, np.inf)
        for k, data in edict.items():
            w = float(data.get("travel_time", np.inf))
            ok = (0.0 + w + hv <= budget)
            if cnt < 10:
                print("orig->", v, "hv", hv, "w", w, "ok", ok)
                cnt += 1

    # keep all feasible edges (all keys) or pick-best; here: keep all keys
    for u in cand_nodes:
        du = dist_o.get(u, np.inf)
        if not np.isfinite(du):
            continue
        for v, edict in G[u].items():
            if v not in cand_set:
                continue
            hv = dist_to_d.get(v, np.inf)
            if not np.isfinite(hv):
                continue
            for k, data in edict.items():
                w_uv = float(data.get("travel_time", np.inf))
                if not np.isfinite(w_uv):
                    continue
                if du + w_uv + hv <= budget:
                    H_gate.add_edge(u, v, key=k, **data)

    print("AFTER BUILD:")
    print("H_gate out-degree orig:", H_gate.out_degree(orig))
    print("H_gate in-degree dest:", H_gate.in_degree(dest))

    # show a few outgoing edges and their travel_time
    sample = list(H_gate.out_edges(orig, keys=True, data=True))[:10]
    for u,v,k,d in sample:
        print("gate edge", u, "->", v, "k", k, "tt", d.get("travel_time"))

    # run dijkstra and inspect reach
    pred_tmp, dist_tmp = nx.dijkstra_predecessor_and_distance(H_gate, orig, weight="travel_time")
    print("dest in dist?", dest in dist_tmp, "dist:", dist_tmp.get(dest))   

    # If the gated graph already disconnects o/d, two-tree can't work under this budget
    if orig not in H_gate or dest not in H_gate:
        return nx.MultiDiGraph()

    # --- B) Dijkstra on H_gate, not on full G ---
    pred_fwd, dist_fwd = nx.dijkstra_predecessor_and_distance(H_gate, orig, weight="travel_time")
    pred_rev, dist_rev = nx.dijkstra_predecessor_and_distance(H_gate.reverse(copy=False), dest, weight="travel_time")

    # --- C') Multi-parent forward structure (from orig outward) ---
    children_fwd = {}
    for v, ps in pred_fwd.items():
        if v == orig:
            continue
        if not isinstance(ps, (list, tuple)):
            ps = [ps]
        for p in ps:
            # p -> v is a tight edge in the forward shortest-path DAG
            children_fwd.setdefault(p, []).append(v)

    # --- D') Multi-parent reverse structure (from dest outward in original direction x -> p) ---
    children_rev = {}
    for x, ps in pred_rev.items():
        if x == dest:
            continue
        if not isinstance(ps, (list, tuple)):
            ps = [ps]
        for p in ps:
            # pred_rev came from H_gate.reverse() rooted at dest:
            # predecessor p for x implies original edge x -> p (toward dest)
            children_rev.setdefault(p, []).append(x)

    print("orig out (fwd children):", len(children_fwd.get(orig, [])))
    print("dest out (rev children):", len(children_rev.get(dest, [])))
    print("has path in H_gate:", nx.has_path(H_gate, orig, dest))

    def pick_parent(pred_map, dist_map, node):
        ps = pred_map.get(node)
        if not ps:
            return None
        if not isinstance(ps, (list, tuple)):
            ps = [ps]
        # deterministic tie-break: smallest dist, then node id
        return min(ps, key=lambda p: (dist_map.get(p, np.inf), p))

    # --- E) Build output graph by meet-in-middle claiming, but edges come from H_gate ---
    H = nx.MultiDiGraph()
    H.graph.update(G.graph)
    add_nodes_with_attrs(H, cand_nodes)

    def add_best_edge_from_gate(u, v):
        # hard DAG guard using the *same* potential everywhere
        pu = dist_fwd.get(u, np.inf)
        pv = dist_fwd.get(v, np.inf)
        if not (np.isfinite(pu) and np.isfinite(pv)):
            return
        if pv <= pu + eps:
            return

        ed = H_gate.get_edge_data(u, v)
        if not ed:
            return
        best_k, best_tt, best_data = None, np.inf, None
        for k, data in ed.items():
            tt = float(data.get("travel_time", np.inf))
            if tt < best_tt:
                best_tt, best_k, best_data = tt, k, data
        if best_data is not None and np.isfinite(best_tt):
            H.add_edge(u, v, key=best_k, **best_data)

    F_claimed, R_claimed = set([orig]), set([dest])
    qF, qR = [orig], [dest]

    meet_edges = f_edges = r_edges = 0

    while qF or qR:
        if qF:
            u = qF.pop()
            for v in children_fwd.get(u, []):
                if v in F_claimed:
                    continue
                if v in R_claimed:
                    add_best_edge_from_gate(u, v)
                    meet_edges += 1
                    continue
                F_claimed.add(v)
                add_best_edge_from_gate(u, v)
                f_edges += 1
                qF.append(v)

        if qR:
            p = qR.pop()
            for x in children_rev.get(p, []):
                if x in R_claimed:
                    continue
                if x in F_claimed:
                    add_best_edge_from_gate(x, p)  # x -> p toward dest
                    meet_edges += 1
                    continue
                R_claimed.add(x)
                add_best_edge_from_gate(x, p)
                r_edges += 1
                qR.append(x)

    print(
        f"Two-tree meet-middle (gated): |F|={len(F_claimed)} |R|={len(R_claimed)} "
        f"edges(fwd={f_edges}, rev={r_edges}, meet={meet_edges})  "
        f"H_gate edges={H_gate.number_of_edges()}"
    )

    H = fill_middle_edges_by_topo(
        H,
        H_gate,
        per_u_cap=8,
        max_forward_span=400,   # tune; None removes this guard
        prefer_fast=True,
    )

    return reachability_prune(H)

def add_lateral_edges_by_do(
    H: nx.MultiDiGraph,
    H_gate: nx.MultiDiGraph,
    *,
    dist_o: dict,
    dist_to_d: dict,
    budget: float,
    eps: float,
    ssp: float,
    jump_frac: float = 0.06,
    keep_per_u: int = 8,
) -> nx.MultiDiGraph:
    """
    Add extra edges among existing H nodes using H_gate as the edge source,
    while preserving acyclicity by enforcing strict increase in d_o.
    """
    if H.number_of_nodes() == 0:
        return H

    core = set(H.nodes)
    max_jump = jump_frac * ssp

    # track how many extra outgoing edges we add per u
    added_out = {u: 0 for u in core}

    # Iterate edges in the gated feasible graph (already budget-feasible by construction)
    for u, v, k, data in H_gate.edges(keys=True, data=True):
        if u not in core or v not in core:
            continue
        if H.has_edge(u, v, key=k):
            continue

        du = dist_o.get(u, np.inf)
        dv = dist_o.get(v, np.inf)
        if not (np.isfinite(du) and np.isfinite(dv)):
            continue

        # DAG guarantee: strict forward progress in d_o
        if dv <= du + eps:
            continue

        # Optional: keep it "local-ish" (prevents huge jumps)
        if (dv - du) > max_jump:
            continue

        # Safety: budget feasibility (in case H_gate wasn’t the source)
        w = float(data.get("travel_time", np.inf))
        hv = dist_to_d.get(v, np.inf)
        if not (np.isfinite(w) and np.isfinite(hv)):
            continue
        if du + w + hv > budget:
            continue

        if added_out[u] >= keep_per_u:
            continue

        H.add_edge(u, v, key=k, **data)
        added_out[u] += 1

    return H

H_orig = build_core_original()
H_tree = build_core_two_tree_middleband()

print("Original core:", H_orig.number_of_nodes(), "nodes,", H_orig.number_of_edges(), "edges")
print("Two-tree core:", H_tree.number_of_nodes(), "nodes,", H_tree.number_of_edges(), "edges")

# ----------------------------
# 5) Visualize on Folium (GeoPandas.explore) + CRS fix is already handled
# ----------------------------
def edges_gdf(H: nx.MultiDiGraph, label: str) -> gpd.GeoDataFrame:
    if H.number_of_edges() == 0:
        return gpd.GeoDataFrame({"rule": [], "geometry": []}, geometry="geometry", crs="EPSG:4326")
    gdf = ox.graph_to_gdfs(H, nodes=False, edges=True, fill_edge_geometry=True).reset_index()
    gdf["rule"] = label
    for col in ["travel_time", "length"]:
        if col not in gdf.columns:
            gdf[col] = np.nan
    return gdf[["u", "v", "key", "rule", "length", "travel_time", "geometry"]]

gdf_o = edges_gdf(H_orig, "original (F monotone)")
gdf_t = edges_gdf(H_tree, "two-tree middle-band")

mid_lat = (y0 + y1) / 2.0
mid_lon = (x0 + x1) / 2.0
m = folium.Map(location=[mid_lat, mid_lon], zoom_start=12, tiles="CartoDB positron")

if len(gdf_o):
    m = gdf_o.explore(m=m, name="original (F)", tooltip=["rule", "travel_time"])
if len(gdf_t):
    m = gdf_t.explore(m=m, name="two-tree middle-band", tooltip=["rule", "travel_time"])

folium.Marker([y0, x0], tooltip="Start (lat,lon)", icon=folium.Icon(color="green")).add_to(m)
folium.Marker([y1, x1], tooltip="End (lat,lon)",   icon=folium.Icon(color="red")).add_to(m)

folium.LayerControl().add_to(m)

m

In [ ]:
nx.is_directed_acyclic_graph(H_tree)

In [ ]:
import networkx as nx
import osmnx as ox

src_node = ox.nearest_nodes(G, x1, y1)

# distances: node -> dist
# paths:     node -> [src, ..., node]
distances, paths = nx.single_source_dijkstra(G.reverse(), src_node, weight="length")

tree_edges = []
for node, path in paths.items():
    if node == src_node or len(path) < 2:
        continue
    parent = path[-2]  # the node right before 'node' on the shortest path

    # If G is a MultiDiGraph, pick an actual existing edge key between (node, parent)
    if G.is_multigraph():
        # choose the key with smallest length (or just take the first key)
        key = min(G[node][parent], key=lambda k: G[node][parent][k].get("length", float("inf")))
        tree_edges.append((node, parent, key))
    else:
        tree_edges.append((node, parent))

G_spt = G.edge_subgraph(tree_edges).copy()

gdf_nodes, gdf_edges = ox.graph_to_gdfs(G_spt)

# distance to target along the reversed-tree direction:
# edge is (u -> v), so u is "farther", v is "closer to src_node"
gdf_edges["dist_to_target"] = gdf_edges.index.get_level_values(1).map(distances)

gdf_edges[gdf_edges["dist_to_target"] < 30000].explore(column="dist_to_target", cmap="viridis_r")

In [ ]:
import numpy as np
import networkx as nx
import geopandas as gpd
import osmnx as ox
import folium

ox.settings.use_cache = True
ox.settings.log_console = True

# ----------------------------
# CONFIG (edit these)
# ----------------------------
place = "Bernalillo County, NM"
network_type = "drive"

# NOTE: you had y,x order in the comment but used y0,x0 variables.
# OSMnx expects X=lon, Y=lat.
y0, x0 = 35.089004161208564, -106.59095403952803   # start (lat, lon)
# y1, x1 = 35.194257447325974, -106.66607311977107   # end   (lat, lon)
y1, x1 = 35.13055817766804, -106.52443543435083

slack = 1.50                 # budget = slack * ssp
eps_rel = 1e-9               # float hygiene
mid_frac = 0.50              # "middle" at 50% of ssp along d_o
mid_band_width_frac = 0.06   # +/- 6% of ssp (small, stable; change if you want)
keep_dag_by_do = True        # apply d_o DAG filter to the SPT-union core
add_lateral = True
lateral_jump_frac = 0.06   # allow edges that advance <= 6% of ssp in d_o
lateral_keep_per_u = 8      # cap extra outgoing edges per node (keeps graph sane)
# ----------------------------

# ----------------------------
# 1) Load graph and pick nodes
# ----------------------------
G = ox.graph_from_place(place, network_type=network_type, simplify=True)

# Ensure travel_time exists
sample_edge = next(iter(G.edges(data=True)))[-1]
if "travel_time" not in sample_edge:
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)

orig = ox.nearest_nodes(G, X=x0, Y=y0)
dest = ox.nearest_nodes(G, X=x1, Y=y1)
print("Origin node:", orig, "Dest node:", dest)

# ----------------------------
# 2) Distances + SSP
# ----------------------------
dist_o = nx.single_source_dijkstra_path_length(G, orig, weight="travel_time")
dist_to_d = nx.single_source_dijkstra_path_length(G.reverse(copy=False), dest, weight="travel_time")

ssp_path = nx.shortest_path(G, orig, dest, weight="travel_time")
ssp = 0.0
for u, v in zip(ssp_path[:-1], ssp_path[1:]):
    ed = G.get_edge_data(u, v)
    best = min(d.get("travel_time", np.inf) for d in ed.values())
    ssp += float(best)

if not np.isfinite(ssp) or ssp <= 0:
    raise RuntimeError("SSP not finite/positive; choose different OD points.")

budget = slack * ssp
eps = eps_rel * ssp
print(f"SSP: {ssp:.1f}s   budget(slack={slack:.3f}): {budget:.1f}s")

# ----------------------------
# 3) Candidate nodes in band: d_o(u)+d_d(u)<=budget
# ----------------------------
cand_nodes = [
    u for u in G.nodes
    if np.isfinite(dist_o.get(u, np.inf))
    and np.isfinite(dist_to_d.get(u, np.inf))
    and (dist_o[u] + dist_to_d[u] <= budget)
]
cand_set = set(cand_nodes)
print("Candidate nodes:", len(cand_nodes), "of", G.number_of_nodes())

F = {u: dist_o[u] - dist_to_d[u] for u in cand_nodes}

def topo_index_of_multidigraph(H: nx.MultiDiGraph) -> dict:
    # topological_sort works on DiGraph; multiedges don't matter for acyclicity
    D = nx.DiGraph()
    D.add_nodes_from(H.nodes)
    D.add_edges_from((u, v) for u, v in H.edges())
    order = list(nx.topological_sort(D))
    return {n: i for i, n in enumerate(order)}

def fill_middle_edges_by_topo(
    H: nx.MultiDiGraph,
    H_gate: nx.MultiDiGraph,
    *,
    per_u_cap: int = 8,          # keep graph sane
    max_forward_span: int = 400, # optional: don't connect super-far in topo
    prefer_fast: bool = True,    # choose best travel_time key per (u,v)
) -> nx.MultiDiGraph:
    """
    Adds extra edges that already exist in H_gate, but only between nodes already in H,
    and only if they go forward in a fixed topological order of H.

    Complexity: O(m_gate_core) where m_gate_core = edges in H_gate induced on H.nodes,
    plus small per-node selection overhead.
    """
    if H.number_of_nodes() == 0:
        return H

    idx = topo_index_of_multidigraph(H)
    core = set(H.nodes)

    # candidates[u] = list of (score, u, v, key, data) where score sorts by desirability
    candidates = {u: [] for u in core}

    # Single scan over feasible (budget-gated) edges, but only among core nodes
    for u, v, k, data in H_gate.edges(keys=True, data=True):
        if u not in core or v not in core:
            continue

        iu, iv = idx[u], idx[v]
        if iv <= iu:
            continue  # topo-forward only -> DAG preserved

        if max_forward_span is not None and (iv - iu) > max_forward_span:
            continue

        if H.has_edge(u, v, key=k):
            continue

        tt = float(data.get("travel_time", np.inf))
        # rank candidates: prefer small travel_time; secondary prefer shorter topo span
        score = (tt, iv - iu)

        candidates[u].append((score, u, v, k, data))

    # Now pick up to per_u_cap edges per u.
    # If prefer_fast, keep only best key per (u,v) among candidates.
    for u, items in candidates.items():
        if not items:
            continue

        # Sort once; typical per-u lists are small-ish
        items.sort(key=lambda t: t[0])

        used_pairs = set()
        added = 0
        for (score, uu, vv, kk, dd) in items:
            if added >= per_u_cap:
                break

            if prefer_fast:
                pair = (uu, vv)
                if pair in used_pairs:
                    continue
                used_pairs.add(pair)

                # pick best key among H_gate(u,v)
                ed = H_gate.get_edge_data(uu, vv)
                if not ed:
                    continue
                best_k, best_tt, best_data = None, np.inf, None
                for k2, d2 in ed.items():
                    tt2 = float(d2.get("travel_time", np.inf))
                    if tt2 < best_tt:
                        best_tt, best_k, best_data = tt2, k2, d2
                if best_data is None or not np.isfinite(best_tt):
                    continue
                if not H.has_edge(uu, vv, key=best_k):
                    H.add_edge(uu, vv, key=best_k, **best_data)
                    added += 1
            else:
                H.add_edge(uu, vv, key=kk, **dd)
                added += 1

    return H

ARTERIAL_CLASSES = {
    "motorway", "motorway_link",
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
}

def highway_class(data) -> str:
    hw = data.get("highway")
    if isinstance(hw, (list, tuple)):
        return str(hw[0])
    if hw is None:
        return ""
    return str(hw)

def best_edge_key_data(Hg: nx.MultiDiGraph, u, v):
    ed = Hg.get_edge_data(u, v)
    if not ed:
        return None, None
    best_k, best_tt, best_data = None, np.inf, None
    for k, d in ed.items():
        tt = float(d.get("travel_time", np.inf))
        if tt < best_tt:
            best_tt, best_k, best_data = tt, k, d
    if best_data is None or not np.isfinite(best_tt):
        return None, None
    return best_k, best_data

def inject_arterials_one_pass(
    H: nx.MultiDiGraph,
    H_gate: nx.MultiDiGraph,
    *,
    dist_o: dict,
    dist_to_d: dict,
    ssp: float,
    budget: float,
    eps: float,
    slack_frac: float = 0.18,     # allow edges within 18% of ssp above shortest
    per_u_cap: int = 4,           # cap how many new "arterial" edges each u can spawn
    require_arterial_tag: bool = True,
) -> nx.MultiDiGraph:
    """
    Add missing arterial/secondary structure by selecting low-slack edges in H_gate
    and growing the node set forward in dist_o. No BFS/DFS passes.

    Complexity: ~O(m_gate) scan + small overhead, bounded edges added.
    """

    if H.number_of_nodes() == 0:
        return H

    core = set(H.nodes)
    slack_max = slack_frac * ssp

    # track added outgoing arterial edges per u (keeps it sparse)
    added_out = {u: 0 for u in core}

    # Collect candidate edges in one scan:
    # We only consider edges already budget-feasible (H_gate), and forward in dist_o.
    cand = []  # (dv, slack, u, v)
    for u, v, k, data in H_gate.edges(keys=True, data=True):
        du = dist_o.get(u, np.inf)
        dv = dist_o.get(v, np.inf)
        hv = dist_to_d.get(v, np.inf)
        if not (np.isfinite(du) and np.isfinite(dv) and np.isfinite(hv)):
            continue
        if dv <= du + eps:
            continue

        if require_arterial_tag:
            hw = highway_class(data)
            if hw not in ARTERIAL_CLASSES:
                continue

        w = float(data.get("travel_time", np.inf))
        if not np.isfinite(w):
            continue

        # (safety) budget feasibility, though H_gate should already satisfy it
        if du + w + hv > budget:
            continue

        slack = du + w + hv - ssp
        if slack < -1e-6:
            # numerical weirdness; clamp
            slack = 0.0

        if slack <= slack_max:
            cand.append((dv, slack, u, v))

    # Grow forward in dist_o of v: ensures we only attach nodes once their "time layer" arrives
    cand.sort(key=lambda t: (t[0], t[1]))

    for dv, slack, u, v in cand:
        if u not in core:
            continue  # only grow outward from already-included nodes

        if added_out.get(u, 0) >= per_u_cap:
            continue

        # pick best key/data for u->v among gated multiedges
        best_k, best_data = best_edge_key_data(H_gate, u, v)
        if best_data is None:
            continue

        # add node if new (must preserve node attrs for gdfs)
        if v not in H:
            H.add_node(v, **G.nodes[v])  # uses your outer-scope G
            core.add(v)
            added_out[v] = 0  # initialize

        if not H.has_edge(u, v, key=best_k):
            H.add_edge(u, v, key=best_k, **best_data)
            added_out[u] += 1

    return H

def inject_arterial_chains(
    H: nx.MultiDiGraph,
    H_gate: nx.MultiDiGraph,
    *,
    G_full: nx.MultiDiGraph,     # for node attrs
    dist_o: dict,
    dist_to_d: dict,
    ssp: float,
    budget: float,
    eps: float,
    slack_frac: float = 0.22,     # max (du+w+hv-ssp) allowed
    chain_len: int = 6,           # how far to “walk” along arterials once we enter them
    per_u_cap: int = 2,           # how many arterial branches each core node can start
    per_layer_cap: int = 80,      # global cap per coarse time-layer (prevents explosion)
    layer_dt_frac: float = 0.03,  # layer size as % of ssp (e.g. 3%)

    # --- new knobs for incremental cycle avoidance ---
    recompute_topo_every: int = 120,   # recompute topo idx after this many "backward-but-safe" insertions
    bounded_dfs_node_cap: int = 6000,  # hard cap visited nodes in bounded DFS to prevent worst-case blowup
) -> nx.MultiDiGraph:
    """
    Grow H by adding short arterial/secondary chains from existing core nodes.

    DAG guarantee:
      We NEVER add an edge that creates a cycle. We use current topo order as a fast filter.
      - If idx[u] < idx[v], edge is safe.
      - Else, we do a bounded reachability test to see if v can reach u.
    """
    if H.number_of_nodes() == 0:
        return H

    slack_max = slack_frac * ssp
    core = set(H.nodes)

    # coarse layering to avoid adding too many edges in the same "time band"
    layer_dt = max(layer_dt_frac * ssp, 1.0)

    def layer_id(u):
        du = dist_o.get(u, np.inf)
        if not np.isfinite(du):
            return None
        return int(du // layer_dt)

    layer_added = {}                 # layer -> count
    started_out = {u: 0 for u in core}

    # ---- topo utilities ----
    def recompute_topo_index() -> dict:
        return topo_index_of_multidigraph(H)

    idx = recompute_topo_index()

    def bounded_reaches(v, u, idx_v, idx_u) -> bool:
        """
        Return True iff there is a path v ~> u in current H, restricted to nodes x with
        idx_v <= idx[x] <= idx_u. (This is sufficient in a DAG.)
        """
        if v == u:
            return True
        if idx_u < idx_v:
            return False

        # iterative DFS with hard node cap
        stack = [v]
        seen = {v}
        visited = 0

        while stack:
            x = stack.pop()
            visited += 1
            if visited > bounded_dfs_node_cap:
                # Safety valve: in worst-case, treat as "reachable" to avoid inserting risky edges
                return True

            # MultiDiGraph: iterate successors
            for y in H.successors(x):
                if y == u:
                    return True
                if y in seen:
                    continue
                iy = idx.get(y)
                if iy is None:
                    continue
                if iy < idx_v or iy > idx_u:
                    continue
                seen.add(y)
                stack.append(y)

        return False

    backward_safe_inserts = 0

    def safe_add_edge(a, b, k, d) -> bool:
        """
        Try to add edge a->b. Return True if added, False if rejected (would create cycle).
        Uses topo filter + bounded DFS only when needed.
        """
        nonlocal idx, backward_safe_inserts

        if H.has_edge(a, b, key=k):
            return True  # already there

        # Adding to a brand-new node cannot create a cycle (no edges out of it yet),
        # but once it's in H we treat it normally.
        ia = idx.get(a)
        ib = idx.get(b)

        # If b is new (no topo index), we can add node and then recompute topo later.
        if ib is None:
            H.add_edge(a, b, key=k, **d)
            # topo index is stale; rebuild now (cheap compared to doing it wrong)
            idx = recompute_topo_index()
            return True

        # If a has no index, something is inconsistent; rebuild topo and retry once.
        if ia is None:
            idx = recompute_topo_index()
            ia = idx.get(a)
            ib = idx.get(b)
            if ia is None or ib is None:
                return False

        # Fast path: forward in topo order => cannot create a cycle
        if ia < ib:
            H.add_edge(a, b, key=k, **d)
            return True

        # Potential back-edge: only safe if b cannot reach a
        if bounded_reaches(b, a, ib, ia):
            return False

        # Safe but topo order is now invalid; add edge and periodically rebuild idx
        H.add_edge(a, b, key=k, **d)
        idx = recompute_topo_index()
        backward_safe_inserts = 0
        return True

    # ---- helpers for choosing arterial edges ----
    def add_node_if_missing(n):
        if n not in H:
            H.add_node(n, **G_full.nodes[n])
            core.add(n)
            started_out[n] = 0
            # topo index needs refresh to include this node
            nonlocal_idx = False  # just for readability

    def best_arterial_out(u):
        """
        Return best arterial outgoing edge (u->v) in H_gate by (slack, travel_time),
        plus its key/data, restricted by budget/slack constraints.
        """
        du = dist_o.get(u, np.inf)
        if not np.isfinite(du) or u not in H_gate:
            return None

        best = None  # (slack, w, dv, u, v, k, data)
        for v, edict in H_gate[u].items():
            hv = dist_to_d.get(v, np.inf)
            if not np.isfinite(hv):
                continue
            for k, d in edict.items():
                if highway_class(d) not in ARTERIAL_CLASSES:
                    continue
                w = float(d.get("travel_time", np.inf))
                if not np.isfinite(w):
                    continue
                if du + w + hv > budget:
                    continue
                slack = du + w + hv - ssp
                if slack < 0:
                    slack = 0.0
                if slack > slack_max:
                    continue
                dv = dist_o.get(v, np.inf)
                cand = (slack, w, dv, u, v, k, d)
                if best is None or cand < best:
                    best = cand
        return best

    # ---- build seed list (same spirit as your version) ----
    seeds = []
    for u in list(core):
        if started_out.get(u, 0) >= per_u_cap:
            continue
        du = dist_o.get(u, np.inf)
        if not np.isfinite(du):
            continue

        # collect best arterial edges out of u (not just one; a few good ones)
        # We do a local scan and keep a small shortlist.
        local = []
        if u in H_gate:
            for v, edict in H_gate[u].items():
                hv = dist_to_d.get(v, np.inf)
                if not np.isfinite(hv):
                    continue
                best_k, best_tt, best_data = None, np.inf, None
                for k, d in edict.items():
                    if highway_class(d) not in ARTERIAL_CLASSES:
                        continue
                    w = float(d.get("travel_time", np.inf))
                    if not np.isfinite(w):
                        continue
                    if du + w + hv > budget:
                        continue
                    slack = du + w + hv - ssp
                    if slack < 0:
                        slack = 0.0
                    if slack > slack_max:
                        continue
                    if w < best_tt:
                        best_tt, best_k, best_data = w, k, d
                if best_data is None:
                    continue
                dv = dist_o.get(v, np.inf)
                local.append((slack, best_tt, dv, u, v, best_k, best_data))

        local.sort(key=lambda t: (t[0], t[1], t[2]))
        # keep a few candidates per u; overall caps still apply
        seeds.extend(local[: max(4, per_u_cap * 2)])

    seeds.sort(key=lambda t: (t[0], t[1], t[2]))

    # ---- inject ----
    for slack, w, dv, u, v, k, d in seeds:
        if u not in core:
            continue
        if started_out.get(u, 0) >= per_u_cap:
            continue

        lid = layer_id(u)
        if lid is None:
            continue
        if layer_added.get(lid, 0) >= per_layer_cap:
            continue

        # Ensure v exists with attrs
        if v not in H:
            H.add_node(v, **G_full.nodes[v])
            core.add(v)
            started_out[v] = 0
            # refresh topo idx so v has an index
            idx = recompute_topo_index()
            backward_safe_inserts = 0

        # Start chain edge u->v with cycle avoidance
        if not safe_add_edge(u, v, k, d):
            continue

        started_out[u] = started_out.get(u, 0) + 1
        layer_added[lid] = layer_added.get(lid, 0) + 1

        # Extend chain: greedy best arterial out of current node
        curr = v
        for _ in range(chain_len - 1):
            if curr == dest:
                break
            nxt = best_arterial_out(curr)
            if nxt is None:
                break
            _sl2, _w2, _dv2, a, b, k2, d2 = nxt

            if b not in H:
                H.add_node(b, **G_full.nodes[b])
                core.add(b)
                started_out[b] = 0
                idx = recompute_topo_index()
                backward_safe_inserts = 0

            if not safe_add_edge(a, b, k2, d2):
                break  # can't extend without cycles; stop this chain

            curr = b
            # If we re-enter the original core, stop early
            if curr in core:
                break

    return H


# ============================================================
# ANGLE-BASED ARTERIAL CHAIN PRECOMPUTE + FAST DAG INJECTION
# ============================================================
# Drop these functions into your current file (below helpers like
# highway_class / ARTERIAL_CLASSES / topo_index_of_multidigraph).
#
# Usage inside build_core_two_tree_middleband():
#   1) after you build H_gate, call:
#        art = precompute_arterial_chains_angle(H_gate, G_full=G, cand_set=cand_set, ...)
#   2) after you build base H (two-tree + fill_middle_edges_by_topo), call:
#        H = inject_arterial_segments_into_dag(H, art, ...)
#
# This avoids per-edge topo recomputes: cycle checks only happen at
# segment endpoints that touch the current DAG, and chains are precomputed once.
# ============================================================

import math
from collections import defaultdict, deque

# ----------------------------
# Small geometry helpers
# ----------------------------
def _xy(G_full: nx.MultiDiGraph, n):
    d = G_full.nodes[n]
    return float(d["x"]), float(d["y"])

def _angle(u_xy, v_xy):
    # angle of vector u->v in radians
    dx = v_xy[0] - u_xy[0]
    dy = v_xy[1] - u_xy[1]
    return math.atan2(dy, dx)

def _angdiff(a, b):
    # smallest absolute difference between angles a and b
    d = abs(a - b)
    if d > math.pi:
        d = 2.0 * math.pi - d
    return d

def _class_rank(hw: str) -> int:
    # smaller is "better"
    order = [
        "motorway", "motorway_link",
        "trunk", "trunk_link",
        "primary", "primary_link",
        "secondary", "secondary_link",
        "tertiary", "tertiary_link",
    ]
    try:
        return order.index(hw)
    except ValueError:
        return 10_000

def _best_arterial_uv(H_gate: nx.MultiDiGraph, u, v):
    """
    Pick best arterial multiedge for (u,v). Returns (k,data) or (None,None).
    """
    ed = H_gate.get_edge_data(u, v)
    if not ed:
        return None, None
    best_k, best_tt, best_d = None, np.inf, None
    for k, d in ed.items():
        hw = highway_class(d)
        if hw not in ARTERIAL_CLASSES:
            continue
        tt = float(d.get("travel_time", np.inf))
        if np.isfinite(tt) and tt < best_tt:
            best_tt, best_k, best_d = tt, k, d
    return best_k, best_d

# ----------------------------
# Build arterial DiGraph view (one edge per (u,v))
# ----------------------------
def build_arterial_digraph(
    H_gate: nx.MultiDiGraph,
    *,
    cand_set: set,
) -> nx.DiGraph:
    """
    Directed arterial-only simple graph on candidate nodes, choosing the best
    multiedge per (u,v). Keeps travel_time/length/highway attrs.
    """
    A = nx.DiGraph()
    A.add_nodes_from(cand_set)

    # iterate unique (u,v) pairs present in H_gate
    for u, v in H_gate.edges():
        if u not in cand_set or v not in cand_set:
            continue
        k, d = _best_arterial_uv(H_gate, u, v)
        if d is None:
            continue
        A.add_edge(u, v, **d)
    return A

# ----------------------------
# Angle-based "next edge" selector at a node
# ----------------------------
def choose_next_by_angle(
    A: nx.DiGraph,
    G_full: nx.MultiDiGraph,
    prev_u,
    curr_v,
    *,
    theta_max_rad: float,
    theta_ambig_rad: float,
    min_len_m: float,
    stop_on_ambiguity: bool,
) -> int | None:
    """
    Given we arrived on prev_u -> curr_v, choose the next node w for curr_v -> w.
    Returns w or None.

    Rules:
      - candidates are outgoing arterial edges curr_v->w
      - require edge length >= min_len_m (if length present)
      - pick smallest angle deviation from incoming heading
      - stop if ambiguous (top-2 too close) when stop_on_ambiguity
      - also reject near-U-turn (implicitly via theta_max)
    """
    if not A.has_node(curr_v):
        return None

    out = list(A.successors(curr_v))
    if not out:
        return None

    # incoming heading
    pu = prev_u
    pv = curr_v
    pu_xy = _xy(G_full, pu)
    pv_xy = _xy(G_full, pv)
    theta_in = _angle(pu_xy, pv_xy)

    scored = []
    for w in out:
        d = A.get_edge_data(curr_v, w)
        if not d:
            continue
        # length gate (prevents tiny stubs)
        L = float(d.get("length", np.inf))
        if np.isfinite(L) and (L < min_len_m):
            continue

        pw_xy = _xy(G_full, w)
        theta_out = _angle(pv_xy, pw_xy)
        da = _angdiff(theta_in, theta_out)

        if da > theta_max_rad:
            continue

        # tie break: class then travel_time/length
        hw = highway_class(d)
        rank = _class_rank(hw)
        tt = float(d.get("travel_time", np.inf))
        scored.append((da, rank, tt, w))

    if not scored:
        return None

    scored.sort()
    if stop_on_ambiguity and len(scored) >= 2:
        if (scored[1][0] - scored[0][0]) <= theta_ambig_rad:
            return None

    return scored[0][3]

# ----------------------------
# Precompute arterial segments (directional chains cut at arterial junctions)
# ----------------------------
def precompute_arterial_chains_angle(
    H_gate: nx.MultiDiGraph,
    *,
    G_full: nx.MultiDiGraph,
    cand_set: set,
    theta_max_deg: float = 25.0,
    theta_ambig_deg: float = 6.0,
    min_len_m: float = 25.0,
    stop_on_ambiguity: bool = True,
) -> dict:
    """
    Precompute directed arterial segments by:
      1) arterial-only directed graph A (one edge per (u,v))
      2) define continuation at v when in_deg==1 and out_deg==1 within A
         AND the next edge is within angle threshold
      3) produce segments (u0->...->uk) cut at arterial junctions and ambiguity stops

    Returns a dict `art` with:
      art["A"]          : nx.DiGraph arterial view
      art["segments"]   : list of segments, each is dict with:
                          - "nodes": [n0,n1,...,nk]
                          - "edges": [(n0,n1),...,(n{k-1},nk)]
                          - "tt"   : sum travel_time along segment
                          - "len"  : sum length along segment
      art["seg_id_by_edge"] : map (u,v) -> segment id
      art["deg_in"], art["deg_out"] : arterial in/out degree in A
    """
    theta_max_rad = math.radians(theta_max_deg)
    theta_ambig_rad = math.radians(theta_ambig_deg)

    A = build_arterial_digraph(H_gate, cand_set=cand_set)
    deg_in = dict(A.in_degree())
    deg_out = dict(A.out_degree())

    # Build a "next edge" map for edges that are eligible to continue through v.
    # We decide continuation based on angle relative to predecessor.
    next_edge = {}  # (u,v) -> (v,w) or None

    # For each directed edge u->v, propose next (v->w) only when v is "linear-ish".
    for u, v in A.edges():
        # If v isn't linear in the arterial subgraph, we stop at v
        if deg_in.get(v, 0) != 1 or deg_out.get(v, 0) != 1:
            next_edge[(u, v)] = None
            continue

        w = choose_next_by_angle(
            A, G_full, u, v,
            theta_max_rad=theta_max_rad,
            theta_ambig_rad=theta_ambig_rad,
            min_len_m=min_len_m,
            stop_on_ambiguity=stop_on_ambiguity,
        )
        if w is None:
            next_edge[(u, v)] = None
        else:
            next_edge[(u, v)] = (v, w)

    # Identify start edges: those not pointed-to by any other next_edge.
    pointed = set(e2 for e2 in next_edge.values() if e2 is not None)
    starts = [(u, v) for (u, v) in A.edges() if (u, v) not in pointed]

    # Extract segments by following next_edge until stop / repeat.
    visited = set()
    segments = []
    seg_id_by_edge = {}

    def edge_cost(u, v):
        d = A.get_edge_data(u, v) or {}
        tt = float(d.get("travel_time", 0.0))
        L = float(d.get("length", 0.0))
        if not np.isfinite(tt):
            tt = 0.0
        if not np.isfinite(L):
            L = 0.0
        return tt, L

    def walk_from(e0):
        (u, v) = e0
        nodes = [u, v]
        edges = [(u, v)]
        tt_sum, L_sum = edge_cost(u, v)

        cur = (u, v)
        while True:
            nxt = next_edge.get(cur)
            if nxt is None:
                break
            if nxt in visited:
                break
            # guard against accidental tiny directed cycles in A
            if nxt[1] in nodes:
                break
            edges.append(nxt)
            nodes.append(nxt[1])
            tt, L = edge_cost(nxt[0], nxt[1])
            tt_sum += tt
            L_sum += L
            cur = nxt

        return nodes, edges, tt_sum, L_sum

    # Walk from starts
    for e0 in starts:
        if e0 in visited:
            continue
        nodes, edges, tt_sum, L_sum = walk_from(e0)
        for e in edges:
            visited.add(e)
        sid = len(segments)
        for e in edges:
            seg_id_by_edge[e] = sid
        segments.append({"nodes": nodes, "edges": edges, "tt": tt_sum, "len": L_sum})

    # Walk any remaining unvisited arterial edges (mostly cycles or isolated bits)
    for e0 in A.edges():
        if e0 in visited:
            continue
        nodes, edges, tt_sum, L_sum = walk_from(e0)
        for e in edges:
            visited.add(e)
        sid = len(segments)
        for e in edges:
            seg_id_by_edge[e] = sid
        segments.append({"nodes": nodes, "edges": edges, "tt": tt_sum, "len": L_sum})

    art = {
        "A": A,
        "segments": segments,
        "seg_id_by_edge": seg_id_by_edge,
        "deg_in": deg_in,
        "deg_out": deg_out,
    }
    return art

# ----------------------------
# Injection: add arterial segments to existing DAG, slicing at DAG touch points
# ----------------------------
def inject_arterial_segments_into_dag(
    H: nx.MultiDiGraph,
    H_gate: nx.MultiDiGraph,
    art: dict,
    *,
    G_full: nx.MultiDiGraph,
    dist_o: dict,
    dist_to_d: dict,
    ssp: float,
    budget: float,
    eps: float,

    # usefulness / realism
    require_two_touch: bool = True,     # reject appendages by default
    min_h_drop_frac: float = 0.005,     # require (h(a)-h(b)) >= this*ssp for accepted pieces
    rho_detour: float = 0.12,           # require w_piece + h(b) <= (1+rho)*h(a)

    # selection / caps
    max_pieces_total: int = 250,
    max_pieces_per_layer: int = 60,
    layer_dt_frac: float = 0.04,
    process_long_first: bool = True,
) -> nx.MultiDiGraph:
    """
    Overlay precomputed arterial segments onto H (a DAG), while preserving DAGness by
    checking cycles ONLY at segment endpoints that touch H.

    Steps:
      - For each precomputed segment S = n0->...->nk, find indices where ni in H.
      - Slice into pieces between consecutive touch points (a...b).
      - Optionally require two touch points (connectors only).
      - For each piece, cycle test: reject iff H has path b ~> a.
        (fast using topo order + bounded DFS)
      - If accepted, add the piece edges (best multiedge key/data from H_gate).

    This avoids per-edge topo recomputes. Cycle checks are per candidate piece.
    """
    if H.number_of_nodes() == 0 or H.number_of_edges() == 0:
        return H

    layer_dt = max(layer_dt_frac * ssp, 1.0)

    def layer_id(u):
        du = dist_o.get(u, np.inf)
        if not np.isfinite(du):
            return None
        return int(du // layer_dt)

    # --- topo index + bounded reach for cycle test ---
    def recompute_idx():
        return topo_index_of_multidigraph(H)

    idx = recompute_idx()

    def bounded_reaches(v, u, idx_v, idx_u, node_cap=5000) -> bool:
        if v == u:
            return True
        if idx_u < idx_v:
            return False
        stack = [v]
        seen = {v}
        visited = 0
        while stack:
            x = stack.pop()
            visited += 1
            if visited > node_cap:
                return True  # conservative
            for y in H.successors(x):
                if y == u:
                    return True
                if y in seen:
                    continue
                iy = idx.get(y)
                if iy is None:
                    continue
                if iy < idx_v or iy > idx_u:
                    continue
                seen.add(y)
                stack.append(y)
        return False

    def creates_cycle_if_add(a, b) -> bool:
        ia = idx.get(a)
        ib = idx.get(b)
        if ia is None or ib is None:
            return True  # shouldn't happen; be safe
        if ia < ib:
            return False
        # possible back-edge: cycle iff b reaches a
        return bounded_reaches(b, a, ib, ia)

    # --- build candidate pieces ---
    pieces = []
    core = set(H.nodes)

    # Precompute: for speed, only consider segments whose nodes are in the budget band
    segments = art["segments"]

    for sid, seg in enumerate(segments):
        nodes = seg["nodes"]
        if len(nodes) < 2:
            continue

        # touch points in current DAG
        touch_idx = [i for i, n in enumerate(nodes) if n in core]
        if require_two_touch and len(touch_idx) < 2:
            continue
        if len(touch_idx) < 2:
            # allow appendages only if you explicitly want them
            continue

        # build pieces between consecutive touch points
        for i0, i1 in zip(touch_idx[:-1], touch_idx[1:]):
            if i1 <= i0:
                continue
            a = nodes[i0]
            b = nodes[i1]

            # compute travel_time sum along this piece (cheap: use H_gate best edges)
            tt_piece = 0.0
            ok = True
            for j in range(i0, i1):
                u, v = nodes[j], nodes[j + 1]
                k, d = best_edge_key_data(H_gate, u, v)
                if d is None:
                    ok = False
                    break
                tt = float(d.get("travel_time", np.inf))
                if not np.isfinite(tt):
                    ok = False
                    break
                tt_piece += tt
            if not ok:
                continue

            # usefulness / realism filters
            ha = dist_to_d.get(a, np.inf)
            hb = dist_to_d.get(b, np.inf)
            if not (np.isfinite(ha) and np.isfinite(hb)):
                continue

            min_h_drop = min_h_drop_frac * ssp
            if (ha - hb) < min_h_drop:
                continue

            if tt_piece + hb > (1.0 + rho_detour) * ha:
                continue

            lid = layer_id(a)
            if lid is None:
                continue

            # score: connectors first, then length
            score = seg["tt"] if process_long_first else 0.0
            # favor bigger progress connectors
            score += 10.0 * (ha - hb)
            pieces.append((score, lid, sid, i0, i1, a, b, tt_piece))

    # process high score first
    pieces.sort(key=lambda t: t[0], reverse=True)

    # caps
    added_total = 0
    added_layer = defaultdict(int)

    # add pieces
    for score, lid, sid, i0, i1, a, b, tt_piece in pieces:
        if added_total >= max_pieces_total:
            break
        if added_layer[lid] >= max_pieces_per_layer:
            continue

        # cycle test on endpoints only (since interior is new edges along a linear segment)
        if creates_cycle_if_add(a, b):
            continue

        # accept: add edges along the piece
        nodes = segments[sid]["nodes"]
        for j in range(i0, i1):
            u, v = nodes[j], nodes[j + 1]
            k, d = best_edge_key_data(H_gate, u, v)
            if d is None:
                break
            if u not in H:
                H.add_node(u, **G_full.nodes[u])
                core.add(u)
            if v not in H:
                H.add_node(v, **G_full.nodes[v])
                core.add(v)
            if not H.has_edge(u, v, key=k):
                H.add_edge(u, v, key=k, **d)

        # refresh topo index because we added a back-connecting segment that could reorder things
        idx = recompute_idx()

        added_total += 1
        added_layer[lid] += 1

    return H

# ----------------------------
# 4) Build core subgraphs (original rule vs "two-tree + middle band" technique)
# ----------------------------
def reachability_prune(H: nx.MultiDiGraph) -> nx.MultiDiGraph:
    """Keep only nodes on some o->d path in H."""
    if orig not in H or dest not in H:
        return nx.MultiDiGraph()
    fwd = set(nx.descendants(H, orig)) | {orig}
    bwd = set(nx.descendants(H.reverse(copy=False), dest)) | {dest}
    core = fwd & bwd
    if orig not in core or dest not in core:
        return nx.MultiDiGraph()
    return H.subgraph(core).copy()

def add_nodes_with_attrs(H: nx.MultiDiGraph, nodes):
    H.add_nodes_from((n, G.nodes[n]) for n in nodes)

def build_core_original() -> nx.MultiDiGraph:
    """Your current gated + F(v)>F(u) + reachability prune. (Now preserves node attrs.)"""
    H = nx.MultiDiGraph()
    H.graph.update(G.graph)  # preserve CRS/metadata

    # IMPORTANT: include x/y node attrs so graph_to_gdfs can build geometries
    add_nodes_with_attrs(H, cand_nodes)

    for u in cand_nodes:
        du = dist_o[u]
        for v, edict in G[u].items():
            if v not in cand_set:
                continue
            hv = dist_to_d.get(v, np.inf)
            if not np.isfinite(hv):
                continue
            for k, data in edict.items():
                w_uv = float(data.get("travel_time", np.inf))
                if not np.isfinite(w_uv):
                    continue
                if du + w_uv + hv > budget:
                    continue
                if F[v] <= F[u] + eps:
                    continue
                H.add_edge(u, v, key=k, **data)

    return reachability_prune(H)

def build_core_two_tree_middleband() -> nx.MultiDiGraph:
    """
    Meet-in-the-middle on the *budget-gated* subgraph, so parents stay inside the core.
    """

    # --- A) Build the slack-gated subgraph on cand_set (NO monotone F filter) ---
    H_gate = nx.MultiDiGraph()
    H_gate.graph.update(G.graph)
    add_nodes_with_attrs(H_gate, cand_nodes)

    art = precompute_arterial_chains_angle(
        H_gate,
        G_full=G,
        cand_set=cand_set,
        theta_max_deg=25.0,
        theta_ambig_deg=6.0,
        min_len_m=25.0,
        stop_on_ambiguity=True,
    )

    # Also check a few outgoing edges from orig in the full G and see why they fail
    cnt = 0
    for v, edict in G[orig].items():
        if v not in cand_set:
            continue
        hv = dist_to_d.get(v, np.inf)
        for k, data in edict.items():
            w = float(data.get("travel_time", np.inf))
            ok = (0.0 + w + hv <= budget)
            if cnt < 10:
                print("orig->", v, "hv", hv, "w", w, "ok", ok)
                cnt += 1

    # keep all feasible edges (all keys) or pick-best; here: keep all keys
    for u in cand_nodes:
        du = dist_o.get(u, np.inf)
        if not np.isfinite(du):
            continue
        for v, edict in G[u].items():
            if v not in cand_set:
                continue
            hv = dist_to_d.get(v, np.inf)
            if not np.isfinite(hv):
                continue
            for k, data in edict.items():
                w_uv = float(data.get("travel_time", np.inf))
                if not np.isfinite(w_uv):
                    continue
                if du + w_uv + hv <= budget:
                    H_gate.add_edge(u, v, key=k, **data)

    print("AFTER BUILD:")
    print("H_gate out-degree orig:", H_gate.out_degree(orig))
    print("H_gate in-degree dest:", H_gate.in_degree(dest))

    # show a few outgoing edges and their travel_time
    sample = list(H_gate.out_edges(orig, keys=True, data=True))[:10]
    for u,v,k,d in sample:
        print("gate edge", u, "->", v, "k", k, "tt", d.get("travel_time"))

    # run dijkstra and inspect reach
    pred_tmp, dist_tmp = nx.dijkstra_predecessor_and_distance(H_gate, orig, weight="travel_time")
    print("dest in dist?", dest in dist_tmp, "dist:", dist_tmp.get(dest))   

    # If the gated graph already disconnects o/d, two-tree can't work under this budget
    if orig not in H_gate or dest not in H_gate:
        return nx.MultiDiGraph()

    # --- B) Dijkstra on H_gate, not on full G ---
    pred_fwd, dist_fwd = nx.dijkstra_predecessor_and_distance(H_gate, orig, weight="travel_time")
    pred_rev, dist_rev = nx.dijkstra_predecessor_and_distance(H_gate.reverse(copy=False), dest, weight="travel_time")

    # --- C') Multi-parent forward structure (from orig outward) ---
    children_fwd = {}
    for v, ps in pred_fwd.items():
        if v == orig:
            continue
        if not isinstance(ps, (list, tuple)):
            ps = [ps]
        for p in ps:
            # p -> v is a tight edge in the forward shortest-path DAG
            children_fwd.setdefault(p, []).append(v)

    # --- D') Multi-parent reverse structure (from dest outward in original direction x -> p) ---
    children_rev = {}
    for x, ps in pred_rev.items():
        if x == dest:
            continue
        if not isinstance(ps, (list, tuple)):
            ps = [ps]
        for p in ps:
            # pred_rev came from H_gate.reverse() rooted at dest:
            # predecessor p for x implies original edge x -> p (toward dest)
            children_rev.setdefault(p, []).append(x)

    print("orig out (fwd children):", len(children_fwd.get(orig, [])))
    print("dest out (rev children):", len(children_rev.get(dest, [])))
    print("has path in H_gate:", nx.has_path(H_gate, orig, dest))

    def pick_parent(pred_map, dist_map, node):
        ps = pred_map.get(node)
        if not ps:
            return None
        if not isinstance(ps, (list, tuple)):
            ps = [ps]
        # deterministic tie-break: smallest dist, then node id
        return min(ps, key=lambda p: (dist_map.get(p, np.inf), p))

    # --- E) Build output graph by meet-in-middle claiming, but edges come from H_gate ---
    H = nx.MultiDiGraph()
    H.graph.update(G.graph)
    add_nodes_with_attrs(H, cand_nodes)

    def add_best_edge_from_gate(u, v):
        if u not in H_gate or v not in H_gate[u]:
            return
        ed = H_gate.get_edge_data(u, v)
        if not ed:
            return
        best_k, best_tt, best_data = None, np.inf, None
        for k, data in ed.items():
            tt = float(data.get("travel_time", np.inf))
            if tt < best_tt:
                best_tt, best_k, best_data = tt, k, data
        if best_data is not None and np.isfinite(best_tt):
            H.add_edge(u, v, key=best_k, **best_data)

    F_claimed, R_claimed = set([orig]), set([dest])
    qF, qR = [orig], [dest]

    meet_edges = f_edges = r_edges = 0

    while qF or qR:
        if qF:
            u = qF.pop()
            for v in children_fwd.get(u, []):
                if v in F_claimed:
                    continue
                if v in R_claimed:
                    add_best_edge_from_gate(u, v)
                    meet_edges += 1
                    continue
                F_claimed.add(v)
                add_best_edge_from_gate(u, v)
                f_edges += 1
                qF.append(v)

        if qR:
            p = qR.pop()
            for x in children_rev.get(p, []):
                if x in R_claimed:
                    continue
                if x in F_claimed:
                    add_best_edge_from_gate(x, p)  # x -> p toward dest
                    meet_edges += 1
                    continue
                R_claimed.add(x)
                add_best_edge_from_gate(x, p)
                r_edges += 1
                qR.append(x)

    print(
        f"Two-tree meet-middle (gated): |F|={len(F_claimed)} |R|={len(R_claimed)} "
        f"edges(fwd={f_edges}, rev={r_edges}, meet={meet_edges})  "
        f"H_gate edges={H_gate.number_of_edges()}"
    )

    H = fill_middle_edges_by_topo(
        H,
        H_gate,
        per_u_cap=8,
        max_forward_span=400,   # tune; None removes this guard
        prefer_fast=True,
    )

    H = inject_arterial_segments_into_dag(
        H, H_gate, art,
        G_full=G,
        dist_o=dist_o,
        dist_to_d=dist_to_d,
        ssp=ssp,
        budget=budget,
        eps=eps,
        require_two_touch=True,      # kills appendages / pseudo-loops
        min_h_drop_frac=0.002,       # tune 0.002–0.02
        rho_detour=0.18,             # tune 0.05–0.20
        max_pieces_total=250,
        max_pieces_per_layer=60,
        layer_dt_frac=0.04,
        process_long_first=True,
    )

    return reachability_prune(H)

def add_lateral_edges_by_do(
    H: nx.MultiDiGraph,
    H_gate: nx.MultiDiGraph,
    *,
    dist_o: dict,
    dist_to_d: dict,
    budget: float,
    eps: float,
    ssp: float,
    jump_frac: float = 0.06,
    keep_per_u: int = 8,
) -> nx.MultiDiGraph:
    """
    Add extra edges among existing H nodes using H_gate as the edge source,
    while preserving acyclicity by enforcing strict increase in d_o.
    """
    if H.number_of_nodes() == 0:
        return H

    core = set(H.nodes)
    max_jump = jump_frac * ssp

    # track how many extra outgoing edges we add per u
    added_out = {u: 0 for u in core}

    # Iterate edges in the gated feasible graph (already budget-feasible by construction)
    for u, v, k, data in H_gate.edges(keys=True, data=True):
        if u not in core or v not in core:
            continue
        if H.has_edge(u, v, key=k):
            continue

        du = dist_o.get(u, np.inf)
        dv = dist_o.get(v, np.inf)
        if not (np.isfinite(du) and np.isfinite(dv)):
            continue

        # DAG guarantee: strict forward progress in d_o
        if dv <= du + eps:
            continue

        # Optional: keep it "local-ish" (prevents huge jumps)
        if (dv - du) > max_jump:
            continue

        # Safety: budget feasibility (in case H_gate wasn’t the source)
        w = float(data.get("travel_time", np.inf))
        hv = dist_to_d.get(v, np.inf)
        if not (np.isfinite(w) and np.isfinite(hv)):
            continue
        if du + w + hv > budget:
            continue

        if added_out[u] >= keep_per_u:
            continue

        H.add_edge(u, v, key=k, **data)
        added_out[u] += 1

    return H

H_orig = build_core_original()
H_tree = build_core_two_tree_middleband()

print("Original core:", H_orig.number_of_nodes(), "nodes,", H_orig.number_of_edges(), "edges")
print("Two-tree core:", H_tree.number_of_nodes(), "nodes,", H_tree.number_of_edges(), "edges")

# ----------------------------
# 5) Visualize on Folium (GeoPandas.explore) + CRS fix is already handled
# ----------------------------
def edges_gdf(H: nx.MultiDiGraph, label: str) -> gpd.GeoDataFrame:
    if H.number_of_edges() == 0:
        return gpd.GeoDataFrame({"rule": [], "geometry": []}, geometry="geometry", crs="EPSG:4326")
    gdf = ox.graph_to_gdfs(H, nodes=False, edges=True, fill_edge_geometry=True).reset_index()
    gdf["rule"] = label
    for col in ["travel_time", "length"]:
        if col not in gdf.columns:
            gdf[col] = np.nan
    return gdf[["u", "v", "key", "rule", "length", "travel_time", "geometry"]]

gdf_o = edges_gdf(H_orig, "original (F monotone)")
gdf_t = edges_gdf(H_tree, "two-tree middle-band")

mid_lat = (y0 + y1) / 2.0
mid_lon = (x0 + x1) / 2.0
m = folium.Map(location=[mid_lat, mid_lon], zoom_start=12, tiles="CartoDB positron")

if len(gdf_o):
    m = gdf_o.explore(m=m, name="original (F)", tooltip=["rule", "travel_time"])
if len(gdf_t):
    m = gdf_t.explore(m=m, name="two-tree middle-band", tooltip=["rule", "travel_time"])

folium.Marker([y0, x0], tooltip="Start (lat,lon)", icon=folium.Icon(color="green")).add_to(m)
folium.Marker([y1, x1], tooltip="End (lat,lon)",   icon=folium.Icon(color="red")).add_to(m)

folium.LayerControl().add_to(m)

m

In [ ]:
nx.is_directed_acyclic_graph(H_tree)

In [ ]:
# ============================================================
# CLEAN CORE + TARGETED "INTERSECTION-SEED" WEBBING (DAG-SAFE)
#   1) build budget-gated graph H_gate
#   2) build two-tree OD core H (union of forward + reverse SPT edges on H_gate)
#   3) select seed nodes in H that look like arterial/collector/motorway/*_link intersections
#   4) run bounded Dijkstra on the induced gated subgraph (nodes already in H),
#      and add the discovered connector-path edges that move forward in H's topo order
#
# Notes:
#   - This never introduces new nodes (only adds edges between existing core nodes),
#     so DAG safety is easy: we only add edges u->v with topo[u] < topo[v].
#   - Bounded Dijkstra (cutoff) + caps keeps it fast.
# ============================================================

import numpy as np
import networkx as nx
import osmnx as ox

# ----------------------------
# helpers: highway tags
# ----------------------------
def _hw(data):
    hw = data.get("highway")
    if isinstance(hw, (list, tuple)) and hw:
        return str(hw[0])
    return "" if hw is None else str(hw)

def _base_hw(hw: str) -> str:
    return hw[:-5] if hw.endswith("_link") else hw

# treat "collector" as tertiary/unclassified (OSM doesn't usually label "collector")
TARGET_HW = {
    "motorway","motorway_link",
    "trunk","trunk_link",
    "primary","primary_link",
    "secondary","secondary_link",
    "tertiary","tertiary_link",
    "unclassified",  # often collector-ish
}

def _best_multiedge_by_tt(Gm: nx.MultiDiGraph, u, v, weight="travel_time"):
    ed = Gm.get_edge_data(u, v)
    if not ed:
        return None, None
    best_k, best_w, best_d = None, np.inf, None
    for k, d in ed.items():
        w = float(d.get(weight, np.inf))
        if np.isfinite(w) and w < best_w:
            best_w, best_k, best_d = w, k, d
    return best_k, best_d

def _digraph_view(Hm: nx.MultiDiGraph) -> nx.DiGraph:
    D = nx.DiGraph()
    D.add_nodes_from(Hm.nodes)
    D.add_edges_from((u, v) for (u, v) in Hm.edges())
    return D

def _topo_index(Hm: nx.MultiDiGraph) -> dict:
    D = _digraph_view(Hm)
    order = list(nx.topological_sort(D))
    return {n: i for i, n in enumerate(order)}

def _reachability_prune(H: nx.MultiDiGraph, o, d) -> nx.MultiDiGraph:
    if o not in H or d not in H:
        return nx.MultiDiGraph()
    fwd = set(nx.descendants(H, o)) | {o}
    bwd = set(nx.descendants(H.reverse(copy=False), d)) | {d}
    core = fwd & bwd
    if o not in core or d not in core:
        return nx.MultiDiGraph()
    return H.subgraph(core).copy()

# ----------------------------
# build gated graph
# ----------------------------
def build_gate_graph(
    G: nx.MultiDiGraph,
    orig,
    dest,
    *,
    slack: float,
    weight: str = "travel_time",
    eps_rel: float = 1e-9,
):
    # distances
    dist_o = nx.single_source_dijkstra_path_length(G, orig, weight=weight)
    dist_to_d = nx.single_source_dijkstra_path_length(G.reverse(copy=False), dest, weight=weight)

    # SSP
    ssp_path = nx.shortest_path(G, orig, dest, weight=weight)
    ssp = 0.0
    for u, v in zip(ssp_path[:-1], ssp_path[1:]):
        k, d = _best_multiedge_by_tt(G, u, v, weight=weight)
        ssp += float(d.get(weight, np.inf))
    if not np.isfinite(ssp) or ssp <= 0:
        raise RuntimeError("SSP not finite/positive; choose different OD points.")

    budget = float(slack * ssp)
    eps = float(eps_rel * ssp)

    # candidate nodes in budget band
    cand = []
    for u in G.nodes:
        du = dist_o.get(u, np.inf)
        hv = dist_to_d.get(u, np.inf)
        if np.isfinite(du) and np.isfinite(hv) and (du + hv <= budget):
            cand.append(u)
    cand_set = set(cand)

    # gated edges among candidate nodes
    H_gate = nx.MultiDiGraph()
    H_gate.graph.update(G.graph)
    H_gate.add_nodes_from((n, G.nodes[n]) for n in cand)

    for u in cand:
        du = dist_o.get(u, np.inf)
        if not np.isfinite(du) or u not in G:
            continue
        for v, edict in G[u].items():
            if v not in cand_set:
                continue
            hv = dist_to_d.get(v, np.inf)
            if not np.isfinite(hv):
                continue
            for k, data in edict.items():
                w = float(data.get(weight, np.inf))
                if not np.isfinite(w):
                    continue
                if du + w + hv <= budget + 1e-9:
                    H_gate.add_edge(u, v, key=k, **data)

    return H_gate, dist_o, dist_to_d, ssp, budget, eps, cand_set

# ----------------------------
# two-tree core on H_gate
# ----------------------------
def build_two_tree_core(
    H_gate: nx.MultiDiGraph,
    orig,
    dest,
    *,
    weight: str = "travel_time",
):
    if orig not in H_gate or dest not in H_gate:
        return nx.MultiDiGraph()

    # Dijkstra predecessor maps on gated graph
    pred_fwd, dist_fwd = nx.dijkstra_predecessor_and_distance(H_gate, orig, weight=weight)
    pred_rev, dist_rev = nx.dijkstra_predecessor_and_distance(H_gate.reverse(copy=False), dest, weight=weight)

    H = nx.MultiDiGraph()
    H.graph.update(H_gate.graph)
    H.add_nodes_from((n, H_gate.nodes[n]) for n in H_gate.nodes)

    # forward SPT edges: p -> v
    for v, ps in pred_fwd.items():
        if v == orig:
            continue
        for p in ps:
            k, d = _best_multiedge_by_tt(H_gate, p, v, weight=weight)
            if d is not None:
                H.add_edge(p, v, key=k, **d)

    # reverse SPT edges (in original direction): x -> p
    # pred_rev came from H_gate.reverse(); predecessor p for x => original edge x -> p
    for x, ps in pred_rev.items():
        if x == dest:
            continue
        for p in ps:
            k, d = _best_multiedge_by_tt(H_gate, x, p, weight=weight)
            if d is not None:
                H.add_edge(x, p, key=k, **d)

    # prune to actual o->d corridor
    H = _reachability_prune(H, orig, dest)
    return H

# ----------------------------
# seed selection: arterial/collector/motorway/*_link intersections
# ----------------------------
def select_intersection_seeds(
    G_full: nx.MultiDiGraph,
    H_core: nx.MultiDiGraph,
    *,
    min_incident: int = 4,
    min_distinct_bases: int = 2,
    require_link_present: bool = False,
):
    """
    Heuristic intersection definition:
      - node is in H_core
      - has >= min_incident incident edges in full graph whose highway is in TARGET_HW
      - has >= min_distinct_bases distinct base classes among those (e.g. primary vs secondary)
      - optionally require at least one *_link incident edge
    """
    seeds = []
    core_nodes = set(H_core.nodes)

    for n in core_nodes:
        if n not in G_full:
            continue

        bases = set()
        has_link = False
        cnt = 0

        # outgoing
        for v, edict in G_full[n].items():
            for k, d in edict.items():
                hw = _hw(d)
                if hw in TARGET_HW:
                    cnt += 1
                    bases.add(_base_hw(hw))
                    if hw.endswith("_link"):
                        has_link = True

        # incoming (MultiDiGraph doesn't store in same dict)
        for u, _, k, d in G_full.in_edges(n, keys=True, data=True):
            hw = _hw(d)
            if hw in TARGET_HW:
                cnt += 1
                bases.add(_base_hw(hw))
                if hw.endswith("_link"):
                    has_link = True

        if cnt >= min_incident and len(bases) >= min_distinct_bases and (has_link or not require_link_present):
            seeds.append(n)

    return seeds

# ----------------------------
# webbing pass: bounded Dijkstra from seeds on induced gated subgraph
# ----------------------------
def web_out_from_seeds(
    H_core: nx.MultiDiGraph,
    H_gate: nx.MultiDiGraph,
    seeds: list,
    *,
    weight: str = "travel_time",
    cutoff_sec: float = 120.0,     # max local search cost from seed
    min_forward_jump: int = 20,     # require topo(target) >= topo(seed)+jump
    max_targets_per_seed: int = 10,
    max_edges_added_total: int = 4000,
):
    """
    Runs bounded Dijkstra from each seed on H_gate induced to H_core.nodes,
    then adds shortest-path edges (from seed to selected targets) into H_core,
    but ONLY edges that go forward in H_core's current topo order.

    This preserves DAGness (topo order remains valid).
    """
    if H_core.number_of_nodes() == 0:
        return H_core

    core_nodes = set(H_core.nodes)
    Hg = H_gate.subgraph(core_nodes).copy()  # search only among existing nodes (cheap + DAG-safe)

    topo = _topo_index(H_core)
    added = 0

    # fast path reconstruction from predecessor dict produced by single_source_dijkstra
    def add_path_edges(path_nodes):
        nonlocal added
        for u, v in zip(path_nodes[:-1], path_nodes[1:]):
            if added >= max_edges_added_total:
                return
            if topo.get(u, -1) >= topo.get(v, -1):
                continue  # would violate DAG order; skip
            k, d = _best_multiedge_by_tt(H_gate, u, v, weight=weight)
            if d is None:
                continue
            if not H_core.has_edge(u, v, key=k):
                H_core.add_edge(u, v, key=k, **d)
                added += 1

    for s in seeds:
        if added >= max_edges_added_total:
            break
        if s not in Hg:
            continue

        # bounded Dijkstra from seed
        dist, pred = nx.single_source_dijkstra(Hg, s, cutoff=cutoff_sec, weight=weight)

        # candidate targets: forward in topo by at least min_forward_jump
        ts = []
        is_ = topo.get(s, None)
        if is_ is None:
            continue

        for t, dt in dist.items():
            it = topo.get(t, None)
            if it is None or it < is_ + min_forward_jump:
                continue
            ts.append((dt, t))

        ts.sort()
        ts = ts[:max_targets_per_seed]

        # add the best few connector paths
        for _, t in ts:
            # reconstruct path from pred (pred[t] is list of predecessors)
            # pred from nx.single_source_dijkstra is a dict: node -> [parents]
            cur = t
            path = [cur]
            while cur != s:
                ps = pred.get(cur)
                if not ps:
                    break
                cur = ps[0]  # any is fine; dist already optimal
                path.append(cur)
                if len(path) > 5000:
                    break
            if path[-1] != s:
                continue
            path.reverse()
            add_path_edges(path)

    return H_core

# ============================================================
# RUN IT
# ============================================================

# 1) Load graph + travel_time
G = ox.graph_from_place(place, network_type=network_type, simplify=True)
sample_edge = next(iter(G.edges(data=True)))[-1]
if "travel_time" not in sample_edge:
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)

orig = ox.nearest_nodes(G, X=x0, Y=y0)
dest = ox.nearest_nodes(G, X=x1, Y=y1)
print("Origin node:", orig, "Dest node:", dest)

# 2) Gate
H_gate, dist_o, dist_to_d, ssp, budget, eps, cand_set = build_gate_graph(
    G, orig, dest,
    slack=slack,
    weight="travel_time",
    eps_rel=eps_rel,
)
print(f"SSP: {ssp:.1f}s   budget(slack={slack:.3f}): {budget:.1f}s")
print("Gated nodes:", H_gate.number_of_nodes(), "Gated edges:", H_gate.number_of_edges())

# 3) Two-tree core
H = build_two_tree_core(H_gate, orig, dest, weight="travel_time")
print("Two-tree core:", H.number_of_nodes(), "nodes,", H.number_of_edges(), "edges")

# 4) Intersection seeds (tune heuristics as needed)
seeds = select_intersection_seeds(
    G, H,
    min_incident=4,
    min_distinct_bases=2,
    require_link_present=False,
)
print("Seed intersections in core:", len(seeds))

# 5) Webbing pass (bounded, DAG-safe)
H_web = web_out_from_seeds(
    H, H_gate, seeds,
    weight="travel_time",
    cutoff_sec=120.0,          # increase to 180-300 if you want more webbing
    min_forward_jump=20,       # increase to reduce clutter; decrease to add more local density
    max_targets_per_seed=10,   # cap per intersection
    max_edges_added_total=4000 # global cap
)

print("Webbed core:", H_web.number_of_nodes(), "nodes,", H_web.number_of_edges(), "edges")

In [ ]:
import numpy as np
import networkx as nx
import osmnx as ox

# ----------------------------
# robust ordering + DAGify
# ----------------------------
def order_key(u, dist_o, dist_to_d):
    return (float(dist_o.get(u, np.inf)), -float(dist_to_d.get(u, np.inf)), int(u))

def dagify_by_order(Hm: nx.MultiDiGraph, dist_o: dict, dist_to_d: dict):
    """
    Make a guaranteed DAG by keeping only edges u->v that go forward in a strict global order:
      (d_o(u), -d_d(u), id(u)) < (d_o(v), -d_d(v), id(v))
    """
    H = nx.MultiDiGraph()
    H.graph.update(Hm.graph)
    H.add_nodes_from((n, Hm.nodes[n]) for n in Hm.nodes)

    removed = 0
    for u, v, k, d in Hm.edges(keys=True, data=True):
        ku = order_key(u, dist_o, dist_to_d)
        kv = order_key(v, dist_o, dist_to_d)
        if ku < kv:
            H.add_edge(u, v, key=k, **d)
        else:
            removed += 1

    # This should now be a DAG (still MultiDiGraph). Verify on DiGraph view.
    D = nx.DiGraph()
    D.add_nodes_from(H.nodes)
    D.add_edges_from((u, v) for u, v in H.edges())
    if not nx.is_directed_acyclic_graph(D):
        # Extremely rare unless dist labels missing/infinite for nodes; fail loudly.
        raise RuntimeError("dagify_by_order failed: still cyclic (likely missing dist labels).")

    return H, removed

def topo_index_from_order(H: nx.MultiDiGraph, dist_o: dict, dist_to_d: dict):
    """
    Since H is DAGified by the same strict order, a topo index consistent with that order
    is just sorting by the order key (no need for nx.topological_sort).
    """
    nodes_sorted = sorted(H.nodes, key=lambda n: order_key(n, dist_o, dist_to_d))
    return {n: i for i, n in enumerate(nodes_sorted)}

# ----------------------------
# seed selection (same as before, but slightly cleaned)
# ----------------------------
TARGET_HW = {
    "motorway","motorway_link","trunk","trunk_link",
    "primary","primary_link","secondary","secondary_link",
    "tertiary","tertiary_link","unclassified",
}

def hw_tag(d):
    hw = d.get("highway")
    if isinstance(hw, (list, tuple)) and hw:
        return str(hw[0])
    return "" if hw is None else str(hw)

def base_hw(hw: str) -> str:
    return hw[:-5] if hw.endswith("_link") else hw

def select_intersection_seeds(G_full: nx.MultiDiGraph, H_core: nx.MultiDiGraph,
                              *, min_incident=4, min_distinct_bases=2, require_link_present=False):
    core = set(H_core.nodes)
    seeds = []
    for n in core:
        cnt = 0
        bases = set()
        has_link = False

        # out edges
        if n in G_full:
            for v, edict in G_full[n].items():
                for k, d in edict.items():
                    hw = hw_tag(d)
                    if hw in TARGET_HW:
                        cnt += 1
                        bases.add(base_hw(hw))
                        if hw.endswith("_link"):
                            has_link = True

        # in edges
        for u, _, k, d in G_full.in_edges(n, keys=True, data=True):
            hw = hw_tag(d)
            if hw in TARGET_HW:
                cnt += 1
                bases.add(base_hw(hw))
                if hw.endswith("_link"):
                    has_link = True

        if cnt >= min_incident and len(bases) >= min_distinct_bases and (has_link or not require_link_present):
            seeds.append(n)
    return seeds

# ----------------------------
# helper: best multiedge for (u,v)
# ----------------------------
def best_edge_uv_by_tt(Gm: nx.MultiDiGraph, u, v, weight="travel_time"):
    ed = Gm.get_edge_data(u, v)
    if not ed:
        return None, None
    best_k, best_w, best_d = None, np.inf, None
    for k, d in ed.items():
        w = float(d.get(weight, np.inf))
        if np.isfinite(w) and w < best_w:
            best_w, best_k, best_d = w, k, d
    return best_k, best_d

# ----------------------------
# webbing pass using bounded Dijkstra on induced gated subgraph
# ----------------------------
def web_out_from_seeds(
    H_core: nx.MultiDiGraph,
    H_gate: nx.MultiDiGraph,
    seeds: list,
    *,
    dist_o: dict,
    dist_to_d: dict,
    weight="travel_time",
    cutoff_sec=120.0,
    min_forward_jump=20,
    max_targets_per_seed=10,
    max_edges_added_total=4000,
):
    if H_core.number_of_nodes() == 0:
        return H_core

    core_nodes = set(H_core.nodes)
    Hg = H_gate.subgraph(core_nodes).copy()  # search only among existing nodes

    topo = topo_index_from_order(H_core, dist_o, dist_to_d)
    added = 0

    def add_path(path_nodes):
        nonlocal added
        for u, v in zip(path_nodes[:-1], path_nodes[1:]):
            if added >= max_edges_added_total:
                return
            if topo.get(u, -1) >= topo.get(v, -1):
                continue  # keep DAG property
            k, d = best_edge_uv_by_tt(H_gate, u, v, weight=weight)
            if d is None:
                continue
            if not H_core.has_edge(u, v, key=k):
                H_core.add_edge(u, v, key=k, **d)
                added += 1

    for s in seeds:
        if added >= max_edges_added_total:
            break
        if s not in Hg:
            continue

        dist, pred = nx.single_source_dijkstra(Hg, s, cutoff=cutoff_sec, weight=weight)

        is_ = topo.get(s, None)
        if is_ is None:
            continue

        targets = []
        for t, dt in dist.items():
            it = topo.get(t, None)
            if it is None or it < is_ + min_forward_jump:
                continue
            targets.append((dt, t))
        targets.sort()
        targets = targets[:max_targets_per_seed]

        for _, t in targets:
            # reconstruct s->t path using predecessor lists
            cur = t
            path = [cur]
            while cur != s:
                ps = pred.get(cur)
                if not ps:
                    break
                cur = ps[0]
                path.append(cur)
                if len(path) > 5000:
                    break
            if path[-1] != s:
                continue
            path.reverse()
            add_path(path)

    return H_core

# ============================================================
# RUN (assumes you already computed: G, orig, dest, H_gate, dist_o, dist_to_d, etc.)
# ============================================================

# --- build your two-tree H exactly as you already do ---
H = build_two_tree_core(H_gate, orig, dest, weight="travel_time")
print("Two-tree core (raw):", H.number_of_nodes(), "nodes,", H.number_of_edges(), "edges")

# --- DAGify it (fixes your error) ---
H, removed = dagify_by_order(H, dist_o, dist_to_d)
print("DAGified two-tree:", H.number_of_nodes(), "nodes,", H.number_of_edges(), "edges",
      "| removed (backward) edges:", removed)

# --- seeds ---
seeds = select_intersection_seeds(
    G, H,
    min_incident=4,
    min_distinct_bases=2,
    require_link_present=False,
)
print("Seed intersections in core:", len(seeds))

# --- webbing ---
H_web = web_out_from_seeds(
    H, H_gate, seeds,
    dist_o=dist_o,
    dist_to_d=dist_to_d,
    weight="travel_time",
    cutoff_sec=120.0,
    min_forward_jump=20,
    max_targets_per_seed=10,
    max_edges_added_total=4000,
)
print("Webbed core:", H_web.number_of_nodes(), "nodes,", H_web.number_of_edges(), "edges")

In [ ]:
import numpy as np
import networkx as nx
import osmnx as ox
from collections import deque

# ============================================================
# 0) small helpers
# ============================================================
def best_edge_uv_by_tt(Gm: nx.MultiDiGraph, u, v, weight="travel_time"):
    ed = Gm.get_edge_data(u, v)
    if not ed:
        return None, None
    best_k, best_w, best_d = None, np.inf, None
    for k, d in ed.items():
        w = float(d.get(weight, np.inf))
        if np.isfinite(w) and w < best_w:
            best_w, best_k, best_d = w, k, d
    return best_k, best_d

def reachability_prune(H: nx.MultiDiGraph, o, d) -> nx.MultiDiGraph:
    if o not in H or d not in H:
        return nx.MultiDiGraph()
    fwd = set(nx.descendants(H, o)) | {o}
    bwd = set(nx.descendants(H.reverse(copy=False), d)) | {d}
    core = fwd & bwd
    if o not in core or d not in core:
        return nx.MultiDiGraph()
    return H.subgraph(core).copy()

def is_dag_multidigraph(H: nx.MultiDiGraph) -> bool:
    D = nx.DiGraph()
    D.add_nodes_from(H.nodes)
    D.add_edges_from((u, v) for (u, v) in H.edges())
    return nx.is_directed_acyclic_graph(D)

def find_one_cycle(H: nx.MultiDiGraph):
    D = nx.DiGraph()
    D.add_nodes_from(H.nodes)
    D.add_edges_from((u, v) for (u, v) in H.edges())
    try:
        cyc = nx.find_cycle(D, orientation="original")
        return [(u, v) for (u, v, _) in cyc]
    except Exception:
        return None

def topo_index(H: nx.MultiDiGraph) -> dict:
    D = nx.DiGraph()
    D.add_nodes_from(H.nodes)
    D.add_edges_from((u, v) for (u, v) in H.edges())
    order = list(nx.topological_sort(D))
    return {n: i for i, n in enumerate(order)}

# ============================================================
# 1) build gated graph
# ============================================================
def build_gate_graph(
    G: nx.MultiDiGraph,
    orig,
    dest,
    *,
    slack: float,
    weight: str = "travel_time",
    eps_rel: float = 1e-9,
):
    dist_o = nx.single_source_dijkstra_path_length(G, orig, weight=weight)
    dist_to_d = nx.single_source_dijkstra_path_length(G.reverse(copy=False), dest, weight=weight)

    ssp_path = nx.shortest_path(G, orig, dest, weight=weight)
    ssp = 0.0
    for u, v in zip(ssp_path[:-1], ssp_path[1:]):
        k, d = best_edge_uv_by_tt(G, u, v, weight=weight)
        ssp += float(d.get(weight, np.inf))
    if not np.isfinite(ssp) or ssp <= 0:
        raise RuntimeError("SSP not finite/positive; choose different OD points.")

    budget = float(slack * ssp)
    eps = float(eps_rel * ssp)

    cand = []
    for u in G.nodes:
        du = dist_o.get(u, np.inf)
        hv = dist_to_d.get(u, np.inf)
        if np.isfinite(du) and np.isfinite(hv) and (du + hv <= budget):
            cand.append(u)
    cand_set = set(cand)

    H_gate = nx.MultiDiGraph()
    H_gate.graph.update(G.graph)
    H_gate.add_nodes_from((n, G.nodes[n]) for n in cand)

    for u in cand:
        du = dist_o.get(u, np.inf)
        if not np.isfinite(du):
            continue
        for v, edict in G[u].items():
            if v not in cand_set:
                continue
            hv = dist_to_d.get(v, np.inf)
            if not np.isfinite(hv):
                continue
            for k, data in edict.items():
                w = float(data.get(weight, np.inf))
                if not np.isfinite(w):
                    continue
                if du + w + hv <= budget + 1e-9:
                    H_gate.add_edge(u, v, key=k, **data)

    return H_gate, dist_o, dist_to_d, ssp, budget, eps, cand_set

# ============================================================
# 2) YOUR two-tree meet-middle (claiming) core
# ============================================================
def build_two_tree_meet_middle_core(
    H_gate: nx.MultiDiGraph,
    orig,
    dest,
    *,
    weight: str = "travel_time",
    tie_break: str = "min_id",
):
    """
    Meet-middle claiming with stopping:
      - forward expansion from orig on children_fwd
      - reverse expansion from dest on children_rev
      - do NOT "push through" nodes already claimed by the other side
      - add only edges used in claiming

    This matches the 'stop when you hit the other side' behavior (not naive union).
    """
    if orig not in H_gate or dest not in H_gate:
        return nx.MultiDiGraph()

    pred_fwd, dist_fwd = nx.dijkstra_predecessor_and_distance(H_gate, orig, weight=weight)
    pred_rev, dist_rev = nx.dijkstra_predecessor_and_distance(H_gate.reverse(copy=False), dest, weight=weight)

    # Build children lists (multi-parent) but we will only CLAIM a node once.
    children_fwd = {}
    for v, ps in pred_fwd.items():
        if v == orig:
            continue
        for p in ps:
            children_fwd.setdefault(p, []).append(v)

    children_rev = {}
    # pred_rev built on reversed graph: predecessor p for x => original edge x -> p
    for x, ps in pred_rev.items():
        if x == dest:
            continue
        for p in ps:
            children_rev.setdefault(p, []).append(x)

    # deterministic parent choice when multiple predecessors exist (only used when adding edge)
    def pick_best_parent_forward(v):
        ps = pred_fwd.get(v) or []
        if not ps:
            return None
        if tie_break == "min_id":
            return min(ps)
        return min(ps, key=lambda p: (dist_fwd.get(p, np.inf), p))

    def pick_best_parent_reverse(x):
        ps = pred_rev.get(x) or []
        if not ps:
            return None
        if tie_break == "min_id":
            return min(ps)
        return min(ps, key=lambda p: (dist_rev.get(p, np.inf), p))

    H = nx.MultiDiGraph()
    H.graph.update(H_gate.graph)
    H.add_nodes_from((n, H_gate.nodes[n]) for n in H_gate.nodes)

    F_claimed = {orig}
    R_claimed = {dest}
    qF = deque([orig])
    qR = deque([dest])

    while qF or qR:
        # forward step
        if qF:
            u = qF.popleft()
            for v in children_fwd.get(u, []):
                if v in F_claimed:
                    continue
                if v in R_claimed:
                    # meet: add u->v but do not claim/expand through v
                    k, d = best_edge_uv_by_tt(H_gate, u, v, weight=weight)
                    if d is not None:
                        H.add_edge(u, v, key=k, **d)
                    continue

                # claim v with its chosen parent edge (which will be some predecessor in pred_fwd)
                p = pick_best_parent_forward(v)
                if p is None:
                    continue
                k, d = best_edge_uv_by_tt(H_gate, p, v, weight=weight)
                if d is not None:
                    H.add_edge(p, v, key=k, **d)
                F_claimed.add(v)
                qF.append(v)

        # reverse step
        if qR:
            p = qR.popleft()
            for x in children_rev.get(p, []):
                if x in R_claimed:
                    continue
                if x in F_claimed:
                    # meet: add x->p but do not claim/expand through x
                    k, d = best_edge_uv_by_tt(H_gate, x, p, weight=weight)
                    if d is not None:
                        H.add_edge(x, p, key=k, **d)
                    continue

                # claim x with its chosen successor-toward-dest edge (x->parent_in_original)
                parent = pick_best_parent_reverse(x)
                if parent is None:
                    continue
                # original edge is x -> parent
                k, d = best_edge_uv_by_tt(H_gate, x, parent, weight=weight)
                if d is not None:
                    H.add_edge(x, parent, key=k, **d)
                R_claimed.add(x)
                qR.append(x)

    # prune to corridor
    H = reachability_prune(H, orig, dest)

    # sanity check DAG; if not, show cycle (so you can see what violated the intended stopping)
    if not is_dag_multidigraph(H):
        cyc = find_one_cycle(H)
        print("WARNING: two-tree meet-middle produced a cycle (showing one directed cycle):")
        print(cyc[:20] if cyc else cyc)

    return H

# ============================================================
# 3) targeted intersection-seed webbing (bounded Dijkstra)
# ============================================================
TARGET_HW = {
    "motorway","motorway_link","trunk","trunk_link",
    "primary","primary_link","secondary","secondary_link",
    "tertiary","tertiary_link","unclassified",
}

def hw_tag(d):
    hw = d.get("highway")
    if isinstance(hw, (list, tuple)) and hw:
        return str(hw[0])
    return "" if hw is None else str(hw)

def base_hw(hw: str) -> str:
    return hw[:-5] if hw.endswith("_link") else hw

def select_intersection_seeds(G_full: nx.MultiDiGraph, H_core: nx.MultiDiGraph,
                              *, min_incident=4, min_distinct_bases=2, require_link_present=False):
    core = set(H_core.nodes)
    seeds = []
    for n in core:
        cnt = 0
        bases = set()
        has_link = False

        # out edges
        if n in G_full:
            for v, edict in G_full[n].items():
                for _, d in edict.items():
                    hw = hw_tag(d)
                    if hw in TARGET_HW:
                        cnt += 1
                        bases.add(base_hw(hw))
                        if hw.endswith("_link"):
                            has_link = True

        # in edges
        for _, _, _, d in G_full.in_edges(n, keys=True, data=True):
            hw = hw_tag(d)
            if hw in TARGET_HW:
                cnt += 1
                bases.add(base_hw(hw))
                if hw.endswith("_link"):
                    has_link = True

        if cnt >= min_incident and len(bases) >= min_distinct_bases and (has_link or not require_link_present):
            seeds.append(n)
    return seeds

from collections import deque

def topo_init_from_dag(H: nx.MultiDiGraph):
    """Initialize a topo order and pos map from an existing DAG MultiDiGraph."""
    D = nx.DiGraph()
    D.add_nodes_from(H.nodes)
    D.add_edges_from((u, v) for (u, v) in H.edges())
    order = list(nx.topological_sort(D))
    pos = {n: i for i, n in enumerate(order)}
    return order, pos

def _bfs_within_interval(adj, start, lo, hi, pos):
    """Forward BFS restricted to nodes whose pos is in [lo, hi]."""
    seen = set()
    q = deque([start])
    while q:
        x = q.popleft()
        if x in seen:
            continue
        px = pos.get(x, None)
        if px is None or px < lo or px > hi:
            continue
        seen.add(x)
        for y in adj.get(x, ()):
            py = pos.get(y, None)
            if py is None or py < lo or py > hi:
                continue
            if y not in seen:
                q.append(y)
    return seen

def topo_add_edge_online(order, pos, adj, radj, u, v):
    """
    Try to add constraint u->v to an existing topological order.
    If possible, updates 'order' and 'pos' in place and returns True.
    If it would create a cycle, returns False (caller should reject edge).
    """
    if u == v:
        return False

    pu = pos.get(u, None)
    pv = pos.get(v, None)

    # New nodes: append at end with stable position
    if pu is None:
        pos[u] = len(order); order.append(u); pu = pos[u]
    if pv is None:
        pos[v] = len(order); order.append(v); pv = pos[v]

    if pu < pv:
        return True  # already consistent

    # We need to reorder nodes in the interval [pv, pu]
    lo, hi = pv, pu

    # F = nodes reachable forward from v within interval
    F = _bfs_within_interval(adj, v, lo, hi, pos)

    # B = nodes that can reach u (i.e. backward reachable from u) within interval
    B = _bfs_within_interval(radj, u, lo, hi, pos)

    # If there's overlap, adding u->v creates a cycle
    # (because v ->* x and x ->* u already exists with x in overlap, plus u->v closes cycle)
    if F & B:
        return False

    # Otherwise, we can reorder by moving F after B within [lo, hi] while preserving internal order.
    interval_nodes = order[lo:hi+1]

    # keep relative order as currently in 'order'
    F_list = [x for x in interval_nodes if x in F]
    B_list = [x for x in interval_nodes if x in B]
    M_list = [x for x in interval_nodes if (x not in F and x not in B)]  # middle untouched

    # New interval: (interval - F) followed by F
    # but (interval - F) must preserve B before others as it already does; we keep current order via B_list + M_list
    new_interval = B_list + M_list + F_list

    # Write back
    order[lo:hi+1] = new_interval
    for i in range(lo, hi+1):
        pos[order[i]] = i

    return True

import heapq

def web_out_from_seeds_halo_dynamic_topo(
    H_core: nx.MultiDiGraph,
    H_gate: nx.MultiDiGraph,
    seeds: list,
    *,
    G_full: nx.MultiDiGraph,
    dist_o: dict,
    dist_to_d: dict,
    weight="travel_time",
    cutoff_sec=180.0,
    min_forward_jump=10,
    max_targets_per_seed=15,
    max_edges_added_total=8000,
):
    """
    Halo webbing with TRUE dynamic topo maintenance:
      - Search on full H_gate
      - STOP whenever you hit any node currently in H_core (except seed)
      - Choose targets that are downstream by rank among ORIGINAL core nodes
      - Reconstruct path and add edges one-by-one
      - Each edge is validated against and merged into a GLOBAL topo order via online update
      - If an edge would create a cycle, reject that target path (do not partially add)
    """
    if H_core.number_of_nodes() == 0:
        return H_core

    # Freeze original core for ranking / downstream test only
    original_core = set(H_core.nodes)

    # Rank among ORIGINAL nodes (your proxy for downstream)
    orig_sorted = sorted(
        original_core,
        key=lambda n: (float(dist_o.get(n, np.inf)), -float(dist_to_d.get(n, np.inf)), int(n))
    )
    rank = {n: i for i, n in enumerate(orig_sorted)}

    # Initialize dynamic topo state from current DAG
    if not is_dag_multidigraph(H_core):
        raise RuntimeError("H_core must start as a DAG for dynamic topo webbing.")
    order, pos = topo_init_from_dag(H_core)

    # Maintain adjacency for online topo updates on the evolving core graph (simple DiGraph view)
    adj = {n: set() for n in H_core.nodes}
    radj = {n: set() for n in H_core.nodes}
    for u, v in H_core.edges():
        adj.setdefault(u, set()).add(v)
        radj.setdefault(v, set()).add(u)

    added = 0

    def ensure_node(n):
        if n not in H_core:
            H_core.add_node(n, **G_full.nodes[n])
        adj.setdefault(n, set())
        radj.setdefault(n, set())
        if n not in pos:
            pos[n] = len(order)
            order.append(n)

    def try_add_edge_safe(u, v):
        """
        Attempt to add u->v into H_core while keeping DAG by online topo update.
        Returns True if committed, False if it would create a cycle.
        """
        nonlocal added
        if added >= max_edges_added_total:
            return False

        ensure_node(u)
        ensure_node(v)

        # First, check if adding the constraint can be accommodated
        ok = topo_add_edge_online(order, pos, adj, radj, u, v)
        if not ok:
            return False  # would introduce a cycle

        # Now commit into adjacency view
        adj[u].add(v)
        radj[v].add(u)

        # Commit the MultiDiGraph edge using your "best edge"
        k, d = best_edge_uv_by_tt(H_gate, u, v, weight=weight)
        if d is None:
            # If there is no actual edge data in H_gate, roll back adjacency constraint
            # (rare if your pred reconstruction is consistent)
            adj[u].remove(v)
            radj[v].remove(u)
            # We *could* also roll back topo order, but simplest is: don't add constraints that can't be realized.
            # In practice, pred edges come from H_gate so d should exist.
            return False

        if not H_core.has_edge(u, v, key=k):
            H_core.add_edge(u, v, key=k, **d)
            added += 1
        return True

    # Webbing loop
    for s in seeds:
        if added >= max_edges_added_total:
            break
        if s not in H_gate:
            continue

        rs = rank.get(s, None)
        if rs is None:
            continue

        dist = {s: 0.0}
        pred = {s: None}
        pq = [(0.0, s)]
        targets = []

        while pq:
            d_u, u = heapq.heappop(pq)
            if d_u > cutoff_sec:
                continue

            for v in H_gate.successors(u):
                if v in dist:
                    continue

                k, d = best_edge_uv_by_tt(H_gate, u, v, weight=weight)
                if d is None:
                    continue
                w = float(d.get(weight, np.inf))
                if not np.isfinite(w):
                    continue

                nd = d_u + w
                if nd > cutoff_sec:
                    continue

                dist[v] = nd
                pred[v] = u

                # stop at any CURRENT DAG node
                if v in H_core and v != s:
                    # accept as target only if it's an ORIGINAL core node and "downstream enough"
                    if v in original_core:
                        rt = rank.get(v)
                        if rt is not None and rt >= rs + min_forward_jump:
                            targets.append((nd, v))
                    continue

                heapq.heappush(pq, (nd, v))

        targets.sort()
        targets = targets[:max_targets_per_seed]

        for _, t in targets:
            if added >= max_edges_added_total:
                break

            # reconstruct path t -> s
            cur = t
            path = [cur]
            while cur != s:
                p = pred.get(cur)
                if p is None:
                    break
                cur = p
                path.append(cur)
                if len(path) > 10000:
                    break

            if path[-1] != s:
                continue
            path.reverse()

            # IMPORTANT: add path edges transactionally
            # If any edge would create a cycle, reject the whole path.
            committed = []
            ok = True
            for u, v in zip(path[:-1], path[1:]):
                if not try_add_edge_safe(u, v):
                    ok = False
                    break
                committed.append((u, v))

            if not ok:
                # Roll back edges we added for this target (both graph edges and adjacency sets).
                # NOTE: Rolling back topo order itself is nontrivial; instead we do a safe strategy:
                #   - remove committed edges from H_core/adj/radj
                #   - then REBUILD topo state from scratch occasionally
                # Given sizes (~thousands), rebuild is fine and keeps correctness.
                for u, v in committed:
                    # remove from adjacency sets
                    if v in adj.get(u, ()):
                        adj[u].remove(v)
                    if u in radj.get(v, ()):
                        radj[v].remove(u)
                    # remove all multiedges u->v we might have added (conservative)
                    if H_core.has_edge(u, v):
                        # remove only edges that exist; if multiple, remove all (fine for “reject path”)
                        keys = list(H_core[u][v].keys())
                        for kk in keys:
                            H_core.remove_edge(u, v, key=kk)

                # Rebuild topo state from current H_core (still a DAG)
                if not is_dag_multidigraph(H_core):
                    # Shouldn't happen because we reject cycle-creating edges; but if it does, expose immediately.
                    cyc = find_one_cycle(H_core)
                    raise RuntimeError(f"Cycle detected despite rejection. Example cycle: {cyc}")

                order, pos = topo_init_from_dag(H_core)
                adj = {n: set() for n in H_core.nodes}
                radj = {n: set() for n in H_core.nodes}
                for uu, vv in H_core.edges():
                    adj[uu].add(vv)
                    radj[vv].add(uu)

    return H_core

# ============================================================
# RUN (uses your already-defined: place, network_type, x0,y0,x1,y1, slack, eps_rel)
# ============================================================
G = ox.graph_from_place(place, network_type=network_type, simplify=True)
sample_edge = next(iter(G.edges(data=True)))[-1]
if "travel_time" not in sample_edge:
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)

orig = ox.nearest_nodes(G, X=x0, Y=y0)
dest = ox.nearest_nodes(G, X=x1, Y=y1)
print("Origin node:", orig, "Dest node:", dest)

H_gate, dist_o, dist_to_d, ssp, budget, eps, cand_set = build_gate_graph(
    G, orig, dest, slack=slack, weight="travel_time", eps_rel=eps_rel
)
print(f"SSP: {ssp:.1f}s   budget(slack={slack:.3f}): {budget:.1f}s")
print("H_gate:", H_gate.number_of_nodes(), "nodes,", H_gate.number_of_edges(), "edges")

H = build_two_tree_meet_middle_core(H_gate, orig, dest, weight="travel_time", tie_break="min_id")
print("Two-tree core (meet-middle):", H.number_of_nodes(), "nodes,", H.number_of_edges(), "edges",
      "| is_dag:", is_dag_multidigraph(H))

seeds = select_intersection_seeds(G, H, min_incident=4, min_distinct_bases=2, require_link_present=False)
print("Seed intersections in core:", len(seeds))

H_keep_old = H.copy()

H_web = web_out_from_seeds_halo_dynamic_topo(
    H, H_gate, seeds,
    G_full=G,
    dist_o=dist_o,
    dist_to_d=dist_to_d,
    cutoff_sec=360,
    min_forward_jump=1,
    max_targets_per_seed=400,
    max_edges_added_total=15000,
)
H_web = reachability_prune(H_web, orig, dest)
print("Webbed core:", H_web.number_of_nodes(), "nodes,", H_web.number_of_edges(), "edges",
      "| is_dag:", is_dag_multidigraph(H_web))

In [ ]:
snnodes, DAG = ox.graph_to_gdfs(H_web, nodes=True, edges=True)
snnodes.loc[[orig, dest]].explore(m=DAG.explore(), color="red")

In [ ]:
e = ox.graph_to_gdfs(H_keep_old, nodes=False)
snnodes.loc[[orig,dest]].explore(m=e.explore(), color="red")